# RoBERTa model selection

This notebook evaluates three article representations (`512`, `MEAN`, `ATTN`) with frozen RoBERTa features, full fine-tuning, and a random-initialized RoBERTa control. Training and model selection use only the training and validation sets.

In [1]:
# 1) Imports, paths and experiment settings
from __future__ import annotations

import copy
import csv
import gc
import hashlib
import inspect
import json
import math
import os
import random
import shutil
import sys
import time
import warnings
from contextlib import contextmanager
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Sequence, Tuple

# Suppress only known benign/noisy messages. Important Python/CUDA/runtime warnings stay visible.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")
# Required by PyTorch for deterministic CUDA/cuBLAS GEMM operations on CUDA >= 10.2.
# This must be set before CUDA/cuBLAS is initialized.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
QUIET_HARMLESS_WARNINGS = True
if QUIET_HARMLESS_WARNINGS:
    warnings.filterwarnings("ignore", message=r".*resume_download.*", category=FutureWarning)
    warnings.filterwarnings("ignore", message=r".*`clean_up_tokenization_spaces`.*", category=FutureWarning)
    warnings.filterwarnings("ignore", message=r".*use_reentrant.*", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support

import torch
from torch import nn
import torch.nn.functional as F

try:
    import transformers
    from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
except ImportError as e:
    raise ImportError(
        "This notebook needs Hugging Face Transformers. In the pytorch_env environment run once:\n"
        "  pip install -U transformers tokenizers safetensors\n"
        "then restart the notebook kernel."
    ) from e

from IPython.display import display, Markdown

# ---------- Project paths ----------
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.experiment_sampling import BalancedEpochSampler, stable_u64
from scripts.modeling_protocol_final import (
    ROBERTA_MODEL_NAME, ROBERTA_MODEL_REVISION,
    TRANSFORMER_SEARCH_MASTER_SEED, TRANSFORMER_CONFIRMATION_MASTER_SEEDS,
    TRANSFORMER_FT_CONFIRMATION_MASTER_SEEDS,
    FE_LEARNING_RATES as FINAL_FE_LEARNING_RATES, FE_WEIGHT_DECAYS as FINAL_FE_WEIGHT_DECAYS,
    FE_MAX_EPOCHS as FINAL_FE_MAX_EPOCHS, FE_PATIENCE as FINAL_FE_PATIENCE,
    FT_LEARNING_RATES as FINAL_FT_LEARNING_RATES, FT_WEIGHT_DECAYS as FINAL_FT_WEIGHT_DECAYS,
    FT_MAX_EPOCHS as FINAL_FT_MAX_EPOCHS, FT_PATIENCE as FINAL_FT_PATIENCE,
)

_sampler_sig = inspect.signature(BalancedEpochSampler.__init__)
for _kw in ("sampler_seed", "global_shuffle_seed"):
    if _kw not in _sampler_sig.parameters:
        raise RuntimeError(
            "scripts/experiment_sampling.py must be the backward-compatible canonical version "
            "with optional sampler_seed= and global_shuffle_seed= keyword overrides."
        )

CORPUS_PATH = PROJECT_ROOT / "frozen_data" / "corpus_frozen.jsonl"
MANIFEST_PATH = PROJECT_ROOT / "experiment_data" / "experiment_manifest.jsonl"
TRAIN_BUCKETS_PATH = PROJECT_ROOT / "experiment_data" / "train_buckets.json"
OUTPUT_DIR = PROJECT_ROOT / "modeling_results" / "roberta_transformer_final"
BASE_MODEL_DIR = OUTPUT_DIR / "base_model_assets"
FROZEN_CACHE_DIR = OUTPUT_DIR / "frozen_feature_cache"

TOPICS = ["topicA", "topicB"]
PORTALS = ["aljazeera", "bbc", "guardian"]
LABEL_TO_ID = {p: i for i, p in enumerate(PORTALS)}
ID_TO_LABEL = {i: p for p, i in LABEL_TO_ID.items()}
TEXT_FIELD = "model_text"

# ---------- Backbone / representation ----------
MODEL_NAME = ROBERTA_MODEL_NAME
MODEL_REVISION = ROBERTA_MODEL_REVISION  # pinned Hugging Face commit
MAX_SEQUENCE_LENGTH = 512
CONTENT_TOKENS_PER_CHUNK = 510
CHUNK_OVERLAP = 0
ATTENTION_HIDDEN_SIZE = 128
HEAD_DROPOUT = 0.1
ATTN_IMPLEMENTATION = "sdpa"  # falls back to the installed Transformers default only if unsupported

ARCHITECTURES = ["512", "MEAN", "ATTN"]

# ---------- Common training randomness ----------
SEARCH_MASTER_SEED = TRANSFORMER_SEARCH_MASTER_SEED
# Keep the original five-seed list as the base/FE seed set so completed FE artifacts
# retain the exact same base protocol hash. FT uses a separate three-seed subset.
CONFIRMATION_MASTER_SEEDS = TRANSFORMER_CONFIRMATION_MASTER_SEEDS
FT_CONFIRMATION_MASTER_SEEDS = TRANSFORMER_FT_CONFIRMATION_MASTER_SEEDS

# ---------- Frozen-feature search ----------
FE_LEARNING_RATES = FINAL_FE_LEARNING_RATES
FE_WEIGHT_DECAYS = FINAL_FE_WEIGHT_DECAYS
FE_MAX_EPOCHS = FINAL_FE_MAX_EPOCHS
FE_PATIENCE = FINAL_FE_PATIENCE
FE_KEEP_PER_ARCH = 2

# ---------- Full fine-tuning search ----------
FT_LEARNING_RATES = FINAL_FT_LEARNING_RATES
FT_WEIGHT_DECAYS = FINAL_FT_WEIGHT_DECAYS
FT_MAX_EPOCHS = FINAL_FT_MAX_EPOCHS
FT_PATIENCE = FINAL_FT_PATIENCE
FT_KEEP_PER_ARCH = 2
FT_WARMUP_RATIO = 0.10
# The completed experiment used direct backpropagation without gradient checkpointing.
FT_GRADIENT_CHECKPOINTING = False

# ---------- Shared optimization ----------
EFFECTIVE_ARTICLE_BATCH_SIZE = 16
EARLY_STOPPING_MIN_DELTA = 0.001
MAX_GRAD_NORM = 1.0
ADAMW_BETAS = (0.9, 0.999)
ADAMW_EPS = 1e-8
USE_AMP = True
REQUIRE_CUDA = True
VERBOSE_EPOCHS = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if REQUIRE_CUDA and DEVICE.type != "cuda":
    raise RuntimeError(
        "CUDA is not available in this Jupyter kernel. Select the WSL pytorch_env kernel with CUDA-enabled PyTorch "
        "before running the transformer experiment."
    )

# Reproducibility defaults.
# Strict mode fails fast if a requested CUDA operation is still nondeterministic,
# rather than silently continuing or flooding the notebook with warnings.
torch.use_deterministic_algorithms(True)
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)


Project root: /home/dorjan/NLP/nlp_portal_project
Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
VRAM GiB: 6.0
PyTorch: 2.5.1
Transformers: 5.16.1


In [2]:
# 2) Shared utilities and compatibility identifiers

def atomic_json_dump(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True, ensure_ascii=False), encoding="utf-8")
    tmp.replace(path)


def atomic_csv_dump(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(path)


def json_load(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def save_figure(fig: plt.Figure, path: Path, dpi: int = 160) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)


def seed32(*parts: Any) -> int:
    h = hashlib.sha256("||".join(map(str, parts)).encode("utf-8")).digest()
    return int.from_bytes(h[:4], "big", signed=False)


def set_training_rng(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed % (2**32 - 1))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def derived_training_seeds(master_seed: int, topic: str) -> Dict[str, int]:
    return {
        "classifier_seed": seed32(master_seed, "classifier-init", topic),
        "attention_seed": seed32(master_seed, "attention-init", topic),
        "torch_seed": seed32(master_seed, "training-rng", topic),
        "sampler_seed": stable_u64(master_seed, "fair-cycle", topic),
        "shuffle_seed": stable_u64(master_seed, "global-shuffle", topic),
    }

for required in [CORPUS_PATH, MANIFEST_PATH, TRAIN_BUCKETS_PATH]:
    if not required.exists():
        raise FileNotFoundError(required)

# These identifiers are kept so the notebook reuses the already-completed run records and caches.
PROTOCOL_HASH = '5fa51702be769922b580121397830bc7c99f9a614e6aca58ff8f850629124ba1'
FT_PROTOCOL_HASH = '3f31db7ebf13f3e6857f609c56aae0cfcf02891b835e5c2dacfe15e18d25c8e3'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Data

Only training and validation article text is loaded in this notebook. The test set is reserved for the separate final-evaluation notebook.

In [3]:
# 3) Load training and validation data
manifest_rows = []
with MANIFEST_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if r.get("experiment_eligible") and r.get("split") in {"train", "validation", "test"}:
            manifest_rows.append(r)
manifest_all = pd.DataFrame(manifest_rows)

expected_counts = {
    ("train", "topicA"): 811,
    ("validation", "topicA"): 272,
    ("test", "topicA"): 281,
    ("train", "topicB"): 721,
    ("validation", "topicB"): 245,
    ("test", "topicB"): 237,
}
actual_counts = manifest_all.groupby(["split", "topic"]).size().to_dict()
assert actual_counts == expected_counts, actual_counts
assert manifest_all["article_id"].is_unique

model_manifest = manifest_all[manifest_all["split"].isin(["train", "validation"])].copy()
allowed_ids = set(model_manifest.article_id)
text_by_id: Dict[str, str] = {}
with CORPUS_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        aid = r.get("article_id")
        if aid in allowed_ids:
            text_by_id[aid] = r[TEXT_FIELD]

assert set(text_by_id) == allowed_ids, f"Missing train/validation texts: {len(allowed_ids - set(text_by_id))}"
model_manifest["text"] = model_manifest.article_id.map(text_by_id)
assert model_manifest.text.notna().all()

# Stable ordering used everywhere downstream.
model_manifest = model_manifest.sort_values(["topic", "split", "article_id"]).reset_index(drop=True)
model_manifest["label_id"] = model_manifest.portal.map(LABEL_TO_ID).astype(int)

TOPIC_DATA: Dict[str, Dict[str, Any]] = {}
for topic in TOPICS:
    train_df = model_manifest[(model_manifest.topic == topic) & (model_manifest.split == "train")].copy().reset_index(drop=True)
    val_df = model_manifest[(model_manifest.topic == topic) & (model_manifest.split == "validation")].copy().reset_index(drop=True)
    assert train_df.balance_weight.notna().all() and val_df.balance_weight.notna().all()
    TOPIC_DATA[topic] = {
        "train": train_df,
        "validation": val_df,
        "train_index_by_id": {aid: i for i, aid in enumerate(train_df.article_id)},
    }

# Exact effective masses frozen by the experimental split.
expected_masses = {
    ("train", "topicA"): 181.0,
    ("validation", "topicA"): 62.0,
    ("train", "topicB"): 109.0,
    ("validation", "topicB"): 37.0,
}
summary_rows = []
for topic in TOPICS:
    for split in ["train", "validation"]:
        df = TOPIC_DATA[topic][split]
        for portal in PORTALS:
            sub = df[df.portal == portal]
            mass = float(sub.balance_weight.sum())
            assert abs(mass - expected_masses[(split, topic)]) < 1e-9, (topic, split, portal, mass)
            summary_rows.append({"topic": topic, "split": split, "portal": portal, "raw_n": len(sub), "effective_weight_mass": mass})
summary_df = pd.DataFrame(summary_rows)
atomic_csv_dump(summary_df, OUTPUT_DIR / "DATA_SPLIT_SUMMARY.csv")
display(summary_df)



,topic,split,portal,raw_n,effective_weight_mass
0,topicA,train,aljazeera,332,181.0
1,topicA,train,bbc,207,181.0
2,topicA,train,guardian,272,181.0
3,topicA,validation,aljazeera,111,62.0
4,topicA,validation,bbc,68,62.0
5,topicA,validation,guardian,93,62.0
6,topicB,train,aljazeera,112,109.0
7,topicB,train,bbc,194,109.0
8,topicB,train,guardian,415,109.0
9,topicB,validation,aljazeera,38,37.0


## Tokenization and article chunks

Articles are tokenized with the pretrained RoBERTa tokenizer. Text longer than one RoBERTa input is split into consecutive non-overlapping chunks of up to 510 content tokens, with no chunk cap.

In [4]:
# 4) Load RoBERTa assets and tokenize train/validation articles

def base_assets_complete(path: Path) -> bool:
    if not path.exists():
        return False
    has_cfg = (path / "config.json").exists()
    has_tok = (path / "tokenizer_config.json").exists()
    has_weights = any((path / n).exists() for n in ["model.safetensors", "pytorch_model.bin"])
    return has_cfg and has_tok and has_weights


def create_base_assets_if_needed() -> Dict[str, Any]:
    source_path = OUTPUT_DIR / "MODEL_SOURCE.json"
    if base_assets_complete(BASE_MODEL_DIR):
        source = json_load(source_path) if source_path.exists() else {"source": "existing_local_assets"}
        return source

    tmp = OUTPUT_DIR / "base_model_assets.tmp"
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True, exist_ok=True)
    print(f"Downloading/loading {MODEL_NAME} ({MODEL_REVISION}) once and saving local assets...")
    try:
        tok = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, use_fast=True)
        try:
            enc = AutoModel.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, attn_implementation=ATTN_IMPLEMENTATION)
            resolved_attn = getattr(enc.config, "_attn_implementation", ATTN_IMPLEMENTATION)
        except (TypeError, ValueError):
            enc = AutoModel.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
            resolved_attn = getattr(enc.config, "_attn_implementation", "library_default")
    except Exception as e:
        raise RuntimeError(
            f"Could not load {MODEL_NAME}. Ensure the WSL environment has internet access on the first run, "
            "or pre-cache the model in Hugging Face."
        ) from e

    tok.save_pretrained(tmp)
    enc.save_pretrained(tmp, safe_serialization=True)
    resolved_commit = getattr(enc.config, "_commit_hash", None) or getattr(tok, "init_kwargs", {}).get("_commit_hash")
    source = {
        "model_name": MODEL_NAME,
        "requested_revision": MODEL_REVISION,
        "resolved_commit_hash_if_available": resolved_commit,
        "resolved_attention_implementation": resolved_attn,
        "tokenizer_class": tok.__class__.__name__,
        "model_class": enc.__class__.__name__,
        "hidden_size": int(enc.config.hidden_size),
        "vocab_size": int(enc.config.vocab_size),
        "model_max_position_embeddings": int(getattr(enc.config, "max_position_embeddings", -1)),
        "saved_local_assets": str(BASE_MODEL_DIR.relative_to(PROJECT_ROOT)),
    }
    del enc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if BASE_MODEL_DIR.exists():
        shutil.rmtree(BASE_MODEL_DIR)
    tmp.replace(BASE_MODEL_DIR)
    atomic_json_dump(source, source_path)
    return source


MODEL_SOURCE = create_base_assets_if_needed()
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, use_fast=True)
assert tokenizer.pad_token_id is not None
assert tokenizer.unk_token_id is not None
assert tokenizer.bos_token_id is not None
assert tokenizer.eos_token_id is not None

# RoBERTa single-sequence inputs are exactly: <s> X </s>.
# Some Transformers 5.x tokenizer classes no longer expose
# build_inputs_with_special_tokens() directly, so construct this fixed
# RoBERTa format explicitly rather than depending on that convenience method.
# This does NOT change the experimental protocol or tokenization semantics.
assert tokenizer.num_special_tokens_to_add(pair=False) == 2
assert CONTENT_TOKENS_PER_CHUNK + 2 <= MAX_SEQUENCE_LENGTH

ROBERTA_BOS_TOKEN_ID = int(tokenizer.bos_token_id)
ROBERTA_EOS_TOKEN_ID = int(tokenizer.eos_token_id)

def add_roberta_single_sequence_special_tokens(content_ids: Sequence[int]) -> List[int]:
    return [ROBERTA_BOS_TOKEN_ID, *[int(x) for x in content_ids], ROBERTA_EOS_TOKEN_ID]

print(MODEL_SOURCE)

# Article token/chunk store. No test article is present here.
ARTICLE_CHUNKS: Dict[str, Dict[str, Any]] = {}
token_rows = []
unk_total = 0
content_total = 0

for row in model_manifest.itertuples(index=False):
    content_ids = tokenizer(row.text, add_special_tokens=False, truncation=False)["input_ids"]
    if len(content_ids) == 0:
        # The corpus should never contain an empty model article; fail instead of manufacturing content.
        raise RuntimeError(f"Empty tokenization for article {row.article_id}")
    chunks = []
    counts = []
    for start in range(0, len(content_ids), CONTENT_TOKENS_PER_CHUNK):
        part = content_ids[start:start + CONTENT_TOKENS_PER_CHUNK]
        seq = add_roberta_single_sequence_special_tokens(part)
        assert len(seq) <= MAX_SEQUENCE_LENGTH
        chunks.append(seq)
        counts.append(len(part))
    assert sum(counts) == len(content_ids)
    ARTICLE_CHUNKS[row.article_id] = {"chunks": chunks, "content_counts": counts, "total_content_tokens": len(content_ids)}
    unk_n = int(sum(tok == tokenizer.unk_token_id for tok in content_ids))
    unk_total += unk_n
    content_total += len(content_ids)
    whitespace_words = max(1, len(row.text.split()))
    token_rows.append({
        "article_id": row.article_id,
        "topic": row.topic,
        "split": row.split,
        "portal": row.portal,
        "content_tokens": len(content_ids),
        "whitespace_words": whitespace_words,
        "tokens_per_whitespace_word": len(content_ids) / whitespace_words,
        "n_chunks": len(chunks),
        "last_chunk_content_tokens": counts[-1],
        "unk_tokens": unk_n,
        "full_coverage": sum(counts) == len(content_ids),
    })

token_stats = pd.DataFrame(token_rows)
assert token_stats.full_coverage.all()
assert len(token_stats) == len(model_manifest)

summary = (
    token_stats.groupby(["topic", "split", "portal"], as_index=False)
    .agg(
        articles=("article_id", "size"),
        mean_tokens=("content_tokens", "mean"),
        median_tokens=("content_tokens", "median"),
        p95_tokens=("content_tokens", lambda s: float(np.quantile(s, 0.95))),
        max_tokens=("content_tokens", "max"),
        mean_chunks=("n_chunks", "mean"),
        p95_chunks=("n_chunks", lambda s: float(np.quantile(s, 0.95))),
        max_chunks=("n_chunks", "max"),
        mean_tokens_per_word=("tokens_per_whitespace_word", "mean"),
        unk_tokens=("unk_tokens", "sum"),
    )
)
atomic_csv_dump(summary, OUTPUT_DIR / "TOKENIZATION_SUMMARY.csv")
display(summary)

fig, ax = plt.subplots(figsize=(8, 4.5))
for topic in TOPICS:
    vals = token_stats[(token_stats.topic == topic) & (token_stats.split == "train")].n_chunks.to_numpy()
    bins = np.arange(0.5, vals.max() + 1.5, 1)
    ax.hist(vals, bins=bins, alpha=0.5, label=topic)
ax.set_xlabel("Chunks per training article")
ax.set_ylabel("Articles")
ax.set_title("RoBERTa full-article chunk counts (train only)")
ax.legend()
save_figure(fig, OUTPUT_DIR / "plots" / "tokenization_train_chunk_counts.png")


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (827 > 512). Running this sequence through the model will result in indexing errors


{'hidden_size': 768, 'model_class': 'RobertaModel', 'model_max_position_embeddings': 514, 'model_name': 'FacebookAI/roberta-base', 'requested_revision': 'e2da8e2f811d1448a5b465c236feacd80ffbac7b', 'resolved_attention_implementation': 'sdpa', 'resolved_commit_hash_if_available': 'e2da8e2f811d1448a5b465c236feacd80ffbac7b', 'saved_local_assets': 'modeling_results/roberta_transformer_final/base_model_assets', 'tokenizer_class': 'RobertaTokenizer', 'vocab_size': 50265}


,topic,split,portal,articles,mean_tokens,median_tokens,p95_tokens,max_tokens,mean_chunks,p95_chunks,max_chunks,mean_tokens_per_word,unk_tokens
0,topicA,train,aljazeera,332,867.451807,789.0,1499.80,2332,2.204819,3.0,5,1.201964,0
1,topicA,train,bbc,207,975.275362,917.0,1584.80,3505,2.410628,4.0,7,1.187730,0
2,topicA,train,guardian,272,985.636029,958.0,1505.05,2947,2.433824,3.0,6,1.191239,0
3,topicA,validation,aljazeera,111,865.423423,782.0,1520.00,1852,2.198198,3.0,4,1.199028,0
4,topicA,validation,bbc,68,941.455882,896.0,1441.05,1655,2.352941,3.0,4,1.185076,0
5,topicA,validation,guardian,93,980.698925,952.0,1472.20,2008,2.397849,3.0,4,1.190416,0
6,topicB,train,aljazeera,112,955.544643,813.5,1965.25,2779,2.357143,4.0,6,1.220251,0
7,topicB,train,bbc,194,907.324742,878.5,1430.95,2279,2.298969,3.0,5,1.192596,0
8,topicB,train,guardian,415,863.653012,809.0,1454.40,5090,2.200000,3.0,10,1.210243,0
9,topicB,validation,aljazeera,38,956.447368,827.5,1717.05,2159,2.342105,4.0,5,1.212253,0


## Metrics and balanced training

Training uses the predefined balanced article sampler. Validation metrics use all validation articles with their frozen balance weights.

In [5]:
# 5) Weighted metrics, checkpoint ordering and sampler coverage

def weighted_metrics_from_predictions(y_true: np.ndarray, y_pred: np.ndarray, sample_weight: np.ndarray) -> Dict[str, Any]:
    labels = np.arange(len(PORTALS))
    w_f1 = f1_score(y_true, y_pred, labels=labels, average="macro", sample_weight=sample_weight, zero_division=0)
    w_acc = accuracy_score(y_true, y_pred, sample_weight=sample_weight)
    raw_f1 = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    raw_acc = accuracy_score(y_true, y_pred)
    p, r, f, support = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, sample_weight=sample_weight, zero_division=0
    )
    return {
        "weighted_macro_f1": float(w_f1),
        "weighted_accuracy": float(w_acc),
        "raw_macro_f1": float(raw_f1),
        "raw_accuracy": float(raw_acc),
        "weighted_precision_by_class": {PORTALS[i]: float(p[i]) for i in labels},
        "weighted_recall_by_class": {PORTALS[i]: float(r[i]) for i in labels},
        "weighted_f1_by_class": {PORTALS[i]: float(f[i]) for i in labels},
        "weighted_support_by_class": {PORTALS[i]: float(support[i]) for i in labels},
        "weighted_confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels, sample_weight=sample_weight).tolist(),
        "raw_confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }


def cross_entropy_from_logits(logits: np.ndarray, y_true: np.ndarray, weights: np.ndarray | None = None) -> float:
    z = logits.astype(np.float64, copy=False)
    zmax = z.max(axis=1, keepdims=True)
    logsumexp = zmax[:, 0] + np.log(np.exp(z - zmax).sum(axis=1))
    losses = logsumexp - z[np.arange(len(y_true)), y_true]
    return float(losses.mean() if weights is None else np.average(losses, weights=weights))


def evaluate_logits(logits: np.ndarray, y: np.ndarray, weights: np.ndarray) -> Dict[str, Any]:
    pred = logits.argmax(axis=1)
    out = weighted_metrics_from_predictions(y, pred, weights)
    out["weighted_cross_entropy"] = cross_entropy_from_logits(logits, y, weights)
    out["raw_cross_entropy"] = cross_entropy_from_logits(logits, y, None)
    return out


def checkpoint_is_better(candidate: Mapping[str, float], incumbent: Mapping[str, float] | None) -> bool:
    if incumbent is None:
        return True
    eps = 1e-12
    if candidate["weighted_macro_f1"] > incumbent["weighted_macro_f1"] + eps:
        return True
    if abs(candidate["weighted_macro_f1"] - incumbent["weighted_macro_f1"]) <= eps:
        if candidate["weighted_accuracy"] > incumbent["weighted_accuracy"] + eps:
            return True
        if abs(candidate["weighted_accuracy"] - incumbent["weighted_accuracy"]) <= eps:
            if candidate["weighted_cross_entropy"] < incumbent["weighted_cross_entropy"] - eps:
                return True
    return False


def confirmation_sort_key(row: Mapping[str, Any]) -> Tuple[float, float, float, int]:
    return (
        -float(row["mean_best_val_weighted_macro_f1"]),
        -float(row["mean_best_val_weighted_accuracy"]),
        float(row["mean_best_val_weighted_cross_entropy"]),
        int(row.get("config_order", 10**9)),
    )


def make_sampler(master_seed: int, topic: str) -> Tuple[BalancedEpochSampler, Dict[str, int]]:
    seeds = derived_training_seeds(master_seed, topic)
    sampler = BalancedEpochSampler(
        TRAIN_BUCKETS_PATH,
        sampler_seed=seeds["sampler_seed"],
        global_shuffle_seed=seeds["shuffle_seed"],
    )
    return sampler, seeds


train_bucket_obj = json_load(TRAIN_BUCKETS_PATH)
_default_sampler = BalancedEpochSampler(TRAIN_BUCKETS_PATH)
assert _default_sampler.base_seed == int(train_bucket_obj["base_seed"])
assert _default_sampler.sampler_seed == _default_sampler.base_seed
assert _default_sampler.global_shuffle_seed == _default_sampler.base_seed

MIN_EPOCHS_BY_TOPIC = {}
for topic in TOPICS:
    coverage = 1
    for b in train_bucket_obj["buckets"]:
        if b["topic"] != topic:
            continue
        m = int(b["per_epoch_per_portal"])
        for portal in PORTALS:
            n = int(b["counts_by_portal"][portal])
            coverage = max(coverage, math.ceil(n / m))
    MIN_EPOCHS_BY_TOPIC[topic] = max(3, coverage)

expected_epoch_sizes = {"topicA": 543, "topicB": 327}
for topic in TOPICS:
    sampler, _ = make_sampler(SEARCH_MASTER_SEED, topic)
    assert sampler.expected_epoch_size(topic) == expected_epoch_sizes[topic]

print("Minimum epochs for full sampler coverage:", MIN_EPOCHS_BY_TOPIC)


Minimum epochs for full sampler coverage: {'topicA': 5, 'topicB': 8}


## Model architectures

`512` uses the first chunk only. `MEAN` combines all chunks with token-count-weighted mean pooling. `ATTN` learns a small content-based score for each chunk on top of the same token-count prior.

In [6]:
# 6) Architecture modules, pooling, padding and optimizer helpers

def _manual_roberta_linear_init(layer: nn.Linear, seed: int, zero_weight: bool = False) -> None:
    devices = [DEVICE.index or 0] if DEVICE.type == "cuda" else []
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(seed)
        if DEVICE.type == "cuda":
            torch.cuda.manual_seed_all(seed)
        with torch.no_grad():
            if zero_weight:
                layer.weight.zero_()
            else:
                layer.weight.normal_(mean=0.0, std=0.02)
            if layer.bias is not None:
                layer.bias.zero_()


class LengthAwareAttentionPool(nn.Module):
    def __init__(self, hidden_size: int, attention_size: int, attention_seed: int):
        super().__init__()
        self.proj = nn.Linear(hidden_size, attention_size)
        self.score = nn.Linear(attention_size, 1, bias=False)
        _manual_roberta_linear_init(self.proj, attention_seed, zero_weight=False)
        # Exact length-prior initialization: learned score_j == 0 initially.
        _manual_roberta_linear_init(self.score, seed32(attention_seed, "score"), zero_weight=True)

    def forward(self, h: torch.Tensor, content_counts: torch.Tensor, mask: torch.Tensor | None = None):
        # h: [B, M, H] or [M, H]
        squeeze = h.ndim == 2
        if squeeze:
            h = h.unsqueeze(0)
            content_counts = content_counts.unsqueeze(0)
            mask = torch.ones_like(content_counts, dtype=torch.bool) if mask is None else mask.unsqueeze(0)
        elif mask is None:
            mask = content_counts > 0
        scores = self.score(torch.tanh(self.proj(h))).squeeze(-1)
        scores = scores + torch.log(content_counts.float().clamp_min(1.0))
        scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
        weights = torch.softmax(scores, dim=-1)
        article = (weights.unsqueeze(-1) * h).sum(dim=-2)
        return (article.squeeze(0), weights.squeeze(0)) if squeeze else (article, weights)


def length_weighted_mean(h: torch.Tensor, content_counts: torch.Tensor, mask: torch.Tensor | None = None):
    squeeze = h.ndim == 2
    if squeeze:
        h = h.unsqueeze(0)
        content_counts = content_counts.unsqueeze(0)
        mask = torch.ones_like(content_counts, dtype=torch.bool) if mask is None else mask.unsqueeze(0)
    elif mask is None:
        mask = content_counts > 0
    w = content_counts.float() * mask.float()
    w = w / w.sum(dim=-1, keepdim=True).clamp_min(1.0)
    article = (w.unsqueeze(-1) * h).sum(dim=-2)
    return (article.squeeze(0), w.squeeze(0)) if squeeze else (article, w)


class ArticleClassifierHead(nn.Module):
    def __init__(self, hidden_size: int, classifier_seed: int, dropout: float = HEAD_DROPOUT):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, len(PORTALS))
        _manual_roberta_linear_init(self.classifier, classifier_seed, zero_weight=False)

    def forward(self, article_h: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.dropout(article_h))


class ArticleHeadModel(nn.Module):
    """Classifier/aggregator used both for cached FE and inside FT."""
    def __init__(self, arch: str, hidden_size: int, seeds: Mapping[str, int]):
        super().__init__()
        if arch not in ARCHITECTURES:
            raise ValueError(arch)
        self.arch = arch
        self.attention_pool = (
            LengthAwareAttentionPool(hidden_size, ATTENTION_HIDDEN_SIZE, seeds["attention_seed"])
            if arch == "ATTN" else None
        )
        self.head = ArticleClassifierHead(hidden_size, seeds["classifier_seed"])

    def aggregate(self, chunk_h: torch.Tensor, content_counts: torch.Tensor | None = None, mask: torch.Tensor | None = None):
        if self.arch == "512":
            return chunk_h, None
        if self.arch == "MEAN":
            return length_weighted_mean(chunk_h, content_counts, mask)
        return self.attention_pool(chunk_h, content_counts, mask)

    def classify_article_embeddings(self, article_h: torch.Tensor) -> torch.Tensor:
        return self.head(article_h)


class FineTuneModel(nn.Module):
    def __init__(self, arch: str, seeds: Mapping[str, int], gradient_checkpointing: bool = FT_GRADIENT_CHECKPOINTING):
        super().__init__()
        self.arch = arch
        self.encoder = load_encoder_from_local_assets()
        if gradient_checkpointing:
            try:
                self.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
            except TypeError:
                self.encoder.gradient_checkpointing_enable()
            if hasattr(self.encoder.config, "use_cache"):
                self.encoder.config.use_cache = False
        self.article_model = ArticleHeadModel(arch, int(self.encoder.config.hidden_size), seeds)

    def encode_chunks(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        # Encoder uses mixed precision; the downstream 768-D representation is explicitly FP32.
        with autocast_encoder():
            out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
            h = out.last_hidden_state[:, 0, :]
        return h.float()


def load_encoder_from_local_assets() -> nn.Module:
    try:
        return AutoModel.from_pretrained(BASE_MODEL_DIR, attn_implementation=ATTN_IMPLEMENTATION)
    except (TypeError, ValueError):
        return AutoModel.from_pretrained(BASE_MODEL_DIR)


@contextmanager
def autocast_encoder():
    enabled = bool(USE_AMP and DEVICE.type == "cuda")
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=enabled):
        yield


def make_grad_scaler(enabled: bool = True):
    enabled = bool(enabled and DEVICE.type == "cuda")
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def collate_token_sequences(seqs: Sequence[Sequence[int]], device: torch.device = DEVICE):
    max_len = max(len(s) for s in seqs)
    if max_len > MAX_SEQUENCE_LENGTH:
        raise ValueError(max_len)
    input_ids = torch.full((len(seqs), max_len), tokenizer.pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((len(seqs), max_len), dtype=torch.long)
    for i, s in enumerate(seqs):
        n = len(s)
        input_ids[i, :n] = torch.tensor(s, dtype=torch.long)
        attention_mask[i, :n] = 1
    return input_ids.to(device), attention_mask.to(device)


def parameter_groups_for_adamw(model: nn.Module, weight_decay: float):
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        lname = name.lower()
        if name.endswith(".bias") or "layernorm.weight" in lname or "layer_norm.weight" in lname:
            no_decay.append(p)
        else:
            decay.append(p)
    return [
        {"params": decay, "weight_decay": float(weight_decay)},
        {"params": no_decay, "weight_decay": 0.0},
    ]


def make_optimizer(model: nn.Module, lr: float, weight_decay: float):
    return torch.optim.AdamW(
        parameter_groups_for_adamw(model, weight_decay),
        lr=float(lr), betas=ADAMW_BETAS, eps=ADAMW_EPS
    )


def logical_batches(ids: Sequence[str], batch_size: int = EFFECTIVE_ARTICLE_BATCH_SIZE):
    for start in range(0, len(ids), batch_size):
        yield list(ids[start:start + batch_size])


def summarize_preclip_grad_norms(norms: Sequence[float]) -> Dict[str, float | int]:
    """Summarize the global gradient norm returned by clip_grad_norm_ *before* clipping."""
    arr = np.asarray(list(norms), dtype=np.float64)
    if arr.size == 0:
        return {
            "optimizer_steps": 0,
            "preclip_grad_norm_mean": float("nan"),
            "preclip_grad_norm_median": float("nan"),
            "preclip_grad_norm_max": float("nan"),
            "grad_clip_count": 0,
            "grad_clip_fraction": float("nan"),
        }
    clipped = arr > (MAX_GRAD_NORM + 1e-12)
    return {
        "optimizer_steps": int(arr.size),
        "preclip_grad_norm_mean": float(arr.mean()),
        "preclip_grad_norm_median": float(np.median(arr)),
        "preclip_grad_norm_max": float(arr.max()),
        "grad_clip_count": int(clipped.sum()),
        "grad_clip_fraction": float(clipped.mean()),
    }


## Execution settings

The completed experiment used the following GPU-safe settings: inference chunk batch 32, FT-512 physical batch 8, and hierarchical chunk microbatch 8. Full fine-tuning used direct backpropagation without gradient checkpointing.

In [7]:
# 7) Execution settings used by the final experiment
INFERENCE_BATCH_SIZE = 32
FT512_PHYSICAL_BATCH_SIZE = 8
FT512_USE_GRADIENT_CHECKPOINTING = False
HIER_TRAIN_METHOD = "direct"
HIER_CHUNK_MICROBATCH_SIZE = 8
HIER_USE_GRADIENT_CHECKPOINTING = False


def cuda_cleanup() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## Frozen RoBERTa features

For FE models, RoBERTa is kept frozen and chunk `<s>` representations are computed once and cached for reuse.

In [8]:
# 8) Build/reuse frozen chunk-embedding cache
FROZEN_CACHE_NPZ = FROZEN_CACHE_DIR / "train_validation_chunk_embeddings.npz"
FROZEN_CACHE_META = FROZEN_CACHE_DIR / "metadata.json"

article_order = model_manifest.article_id.tolist()
article_order_hash = "c0cda92f057e0a292e9b8ace513bef729417b3c85e74a28006984904acb03f6e"


def build_frozen_feature_cache() -> None:
    FROZEN_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    flat_seqs = []
    flat_counts = []
    offsets = [0]
    for aid in article_order:
        flat_seqs.extend(ARTICLE_CHUNKS[aid]["chunks"])
        flat_counts.extend(ARTICLE_CHUNKS[aid]["content_counts"])
        offsets.append(len(flat_seqs))

    encoder = load_encoder_from_local_assets().to(DEVICE).eval()
    all_h = np.empty((len(flat_seqs), int(encoder.config.hidden_size)), dtype=np.float32)
    t0 = time.time()
    with torch.inference_mode():
        for start in range(0, len(flat_seqs), INFERENCE_BATCH_SIZE):
            seqs = flat_seqs[start:start + INFERENCE_BATCH_SIZE]
            ids, mask = collate_token_sequences(seqs)
            with autocast_encoder():
                out = encoder(input_ids=ids, attention_mask=mask, return_dict=True).last_hidden_state[:, 0, :]
            all_h[start:start + len(seqs)] = out.float().cpu().numpy()
            if start % max(INFERENCE_BATCH_SIZE * 50, 1) == 0:
                print(f"frozen extraction: {start}/{len(flat_seqs)} chunks")
    elapsed = time.time() - t0
    np.savez_compressed(
        FROZEN_CACHE_NPZ,
        chunk_embeddings=all_h,
        chunk_content_counts=np.asarray(flat_counts, dtype=np.int32),
        article_offsets=np.asarray(offsets, dtype=np.int64),
        article_ids=np.asarray(article_order, dtype="U64"),
    )
    meta = {
        "protocol_hash": PROTOCOL_HASH,
        "article_order_hash": article_order_hash,
        "n_articles": len(article_order),
        "n_chunks": len(flat_seqs),
        "hidden_size": int(all_h.shape[1]),
        "dtype": str(all_h.dtype),
        "inference_batch_size": INFERENCE_BATCH_SIZE,
        "seconds": elapsed,
        "test_embeddings_present": False,
    }
    atomic_json_dump(meta, FROZEN_CACHE_META)
    del encoder, all_h
    cuda_cleanup()


if FROZEN_CACHE_NPZ.exists() and FROZEN_CACHE_META.exists():
    meta = json_load(FROZEN_CACHE_META)
    if meta.get("protocol_hash") != PROTOCOL_HASH or meta.get("article_order_hash") != article_order_hash:
        raise RuntimeError("Existing frozen feature cache does not match this protocol/article order.")
else:
    build_frozen_feature_cache()

cache = np.load(FROZEN_CACHE_NPZ, allow_pickle=False)
FROZEN_CHUNK_H = np.asarray(cache["chunk_embeddings"], dtype=np.float32)
FROZEN_CHUNK_COUNTS = np.asarray(cache["chunk_content_counts"], dtype=np.int32)
FROZEN_OFFSETS = np.asarray(cache["article_offsets"], dtype=np.int64)
FROZEN_ARTICLE_IDS = cache["article_ids"].astype(str).tolist()
assert FROZEN_ARTICLE_IDS == article_order
FROZEN_ARTICLE_INDEX = {aid: i for i, aid in enumerate(FROZEN_ARTICLE_IDS)}
print("Frozen cache:", FROZEN_CHUNK_H.shape, "chunks; test embeddings present: False")


Frozen cache: (4692, 768) chunks; test embeddings present: False


In [9]:
# 9) Frozen-feature batching, evaluation and training

def frozen_article_chunk_arrays(aid: str) -> Tuple[np.ndarray, np.ndarray]:
    i = FROZEN_ARTICLE_INDEX[aid]
    a, b = int(FROZEN_OFFSETS[i]), int(FROZEN_OFFSETS[i + 1])
    return FROZEN_CHUNK_H[a:b], FROZEN_CHUNK_COUNTS[a:b]


def frozen_forward_batch(model: ArticleHeadModel, arch: str, aids: Sequence[str]) -> Tuple[torch.Tensor, List[np.ndarray] | None]:
    if arch == "512":
        h = np.stack([frozen_article_chunk_arrays(a)[0][0] for a in aids], axis=0)
        h_t = torch.from_numpy(h).to(DEVICE)
        logits = model.classify_article_embeddings(h_t)
        return logits, None

    chunks = [frozen_article_chunk_arrays(a)[0] for a in aids]
    counts = [frozen_article_chunk_arrays(a)[1] for a in aids]
    max_m = max(len(x) for x in chunks)
    H = chunks[0].shape[1]
    h_pad = np.zeros((len(aids), max_m, H), dtype=np.float32)
    c_pad = np.zeros((len(aids), max_m), dtype=np.float32)
    mask = np.zeros((len(aids), max_m), dtype=bool)
    for i, (h, c) in enumerate(zip(chunks, counts)):
        m = len(h); h_pad[i, :m] = h; c_pad[i, :m] = c; mask[i, :m] = True
    h_t = torch.from_numpy(h_pad).to(DEVICE)
    c_t = torch.from_numpy(c_pad).to(DEVICE)
    m_t = torch.from_numpy(mask).to(DEVICE)
    article_h, weights = model.aggregate(h_t, c_t, m_t)
    logits = model.classify_article_embeddings(article_h)
    weight_list = [weights[i, :len(chunks[i])].detach().float().cpu().numpy() for i in range(len(aids))]
    return logits, weight_list


def evaluate_frozen(model: ArticleHeadModel, arch: str, df: pd.DataFrame, batch_size: int = 128):
    model.eval()
    logits_parts = []
    attention_rows = []
    with torch.inference_mode():
        aids_all = df.article_id.tolist()
        for start in range(0, len(aids_all), batch_size):
            aids = aids_all[start:start + batch_size]
            logits, weights = frozen_forward_batch(model, arch, aids)
            logits_parts.append(logits.float().cpu().numpy())
            if arch == "ATTN" and weights is not None:
                for aid, w in zip(aids, weights):
                    counts = ARTICLE_CHUNKS[aid]["content_counts"]
                    for j, (ww, cc) in enumerate(zip(w, counts)):
                        attention_rows.append({"article_id": aid, "chunk_index": j, "content_tokens": cc, "attention_weight": float(ww)})
    logits = np.concatenate(logits_parts, axis=0)
    y = df.label_id.to_numpy(np.int64)
    weights = df.balance_weight.to_numpy(np.float64)
    return logits, evaluate_logits(logits, y, weights), attention_rows


def probabilities_from_logits(logits: np.ndarray) -> np.ndarray:
    z = logits.astype(np.float64)
    z -= z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def save_predictions(df: pd.DataFrame, logits: np.ndarray, path: Path) -> None:
    probs = probabilities_from_logits(logits)
    pred = logits.argmax(axis=1)
    out = pd.DataFrame({
        "article_id": df.article_id.to_numpy(),
        "topic": df.topic.to_numpy(),
        "portal": df.portal.to_numpy(),
        "y_true": df.label_id.to_numpy(np.int64),
        "y_pred": pred,
        "balance_weight": df.balance_weight.to_numpy(np.float64),
    })
    for i, p in enumerate(PORTALS):
        out[f"logit_{p}"] = logits[:, i]
        out[f"prob_{p}"] = probs[:, i]
    atomic_csv_dump(out, path)


def cpu_state_dict(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def save_torch_state_atomic(state: Mapping[str, torch.Tensor], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(dict(state), tmp)
    tmp.replace(path)


def load_run_record_if_complete(
    record_path: Path, checkpoint_required: bool = False, expected_protocol_hash: str = PROTOCOL_HASH
) -> Dict[str, Any] | None:
    if not record_path.exists():
        return None
    rec = json_load(record_path)
    if rec.get("protocol_hash") != expected_protocol_hash:
        raise RuntimeError(
            f"Run record protocol mismatch: {record_path}; "
            f"expected {expected_protocol_hash}, got {rec.get('protocol_hash')}"
        )
    if checkpoint_required:
        cp = rec.get("checkpoint_path")
        if not cp or not (PROJECT_ROOT / cp).exists():
            return None
    return rec


def train_frozen_run(
    topic: str, arch: str, lr: float, weight_decay: float, master_seed: int,
    run_id: str, out_dir: Path, save_checkpoint: bool,
) -> Dict[str, Any]:
    record_path = out_dir / "run_records" / f"{run_id}.json"
    checkpoint_path = out_dir / "checkpoints" / f"{run_id}.pt" if save_checkpoint else None
    existing = load_run_record_if_complete(record_path, checkpoint_required=save_checkpoint)
    if existing is not None:
        return existing

    train_df = TOPIC_DATA[topic]["train"]
    val_df = TOPIC_DATA[topic]["validation"]
    train_y_by_id = train_df.set_index("article_id").label_id.to_dict()
    sampler, seeds = make_sampler(master_seed, topic)
    set_training_rng(seeds["torch_seed"])
    model = ArticleHeadModel(arch, int(MODEL_SOURCE["hidden_size"]), seeds).to(DEVICE)
    # Reset stochastic training stream AFTER seeded module initialization.
    set_training_rng(seeds["torch_seed"])
    optimizer = make_optimizer(model, lr, weight_decay)

    history = []
    best_metrics = None; best_state = None; best_epoch = None
    meaningful_anchor = -np.inf; epochs_since_meaningful = 0
    all_grad_norms: List[float] = []
    t0 = time.time()
    for epoch in range(FE_MAX_EPOCHS):
        model.train()
        ordered_ids = sampler.article_ids_for_epoch(epoch, topic)
        sampled_loss_sum = 0.0; sampled_n = 0
        epoch_grad_norms: List[float] = []
        for aids in logical_batches(ordered_ids):
            optimizer.zero_grad(set_to_none=True)
            logits, _ = frozen_forward_batch(model, arch, aids)
            y = torch.tensor([train_y_by_id[a] for a in aids], dtype=torch.long, device=DEVICE)
            loss = F.cross_entropy(logits, y)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite FE loss: {run_id}, epoch {epoch+1}")
            loss.backward()
            grad_norm_t = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            if not torch.isfinite(grad_norm_t):
                raise FloatingPointError(f"Non-finite FE gradient norm: {run_id}, epoch {epoch+1}")
            grad_norm = float(grad_norm_t.detach().cpu())
            epoch_grad_norms.append(grad_norm)
            all_grad_norms.append(grad_norm)
            optimizer.step()
            sampled_loss_sum += float(loss.detach().cpu()) * len(aids)
            sampled_n += len(aids)

        train_logits, train_eval, _ = evaluate_frozen(model, arch, train_df)
        val_logits, val_eval, _ = evaluate_frozen(model, arch, val_df)
        epoch_grad_stats = summarize_preclip_grad_norms(epoch_grad_norms)
        row = {
            "epoch": epoch + 1,
            "sampled_train_loss": sampled_loss_sum / sampled_n,
            "full_train_weighted_macro_f1": train_eval["weighted_macro_f1"],
            "full_train_weighted_accuracy": train_eval["weighted_accuracy"],
            "full_train_weighted_cross_entropy": train_eval["weighted_cross_entropy"],
            "full_train_raw_macro_f1": train_eval["raw_macro_f1"],
            "val_weighted_macro_f1": val_eval["weighted_macro_f1"],
            "val_weighted_accuracy": val_eval["weighted_accuracy"],
            "val_weighted_cross_entropy": val_eval["weighted_cross_entropy"],
            "val_raw_macro_f1": val_eval["raw_macro_f1"],
            "val_raw_accuracy": val_eval["raw_accuracy"],
            **epoch_grad_stats,
        }
        history.append(row)
        if checkpoint_is_better(val_eval, best_metrics):
            best_metrics = copy.deepcopy(val_eval); best_epoch = epoch + 1
            if save_checkpoint:
                best_state = cpu_state_dict(model)
        current_f1 = val_eval["weighted_macro_f1"]
        if current_f1 > meaningful_anchor + EARLY_STOPPING_MIN_DELTA:
            meaningful_anchor = current_f1; epochs_since_meaningful = 0
        else:
            epochs_since_meaningful += 1
        if VERBOSE_EPOCHS:
            print(run_id, "epoch", epoch+1, "val_wF1", round(current_f1, 4), "best_wF1", round(best_metrics["weighted_macro_f1"], 4))
        if epoch + 1 >= MIN_EPOCHS_BY_TOPIC[topic] and epochs_since_meaningful >= FE_PATIENCE:
            break

    hist_path = out_dir / "histories" / f"{run_id}.csv"
    atomic_csv_dump(pd.DataFrame(history), hist_path)
    if save_checkpoint:
        assert best_state is not None
        save_torch_state_atomic(best_state, checkpoint_path)
        model.load_state_dict(best_state)
        val_logits, val_eval_check, attn_rows = evaluate_frozen(model, arch, val_df)
        assert abs(val_eval_check["weighted_macro_f1"] - best_metrics["weighted_macro_f1"]) < 1e-10
        pred_path = out_dir / "validation_predictions" / f"{run_id}.csv"
        save_predictions(val_df, val_logits, pred_path)
        if arch == "ATTN":
            atomic_csv_dump(pd.DataFrame(attn_rows), out_dir / "attention_weights" / f"{run_id}.csv")
    elapsed = time.time() - t0
    run_grad_stats = summarize_preclip_grad_norms(all_grad_norms)
    rec = {
        "protocol_hash": PROTOCOL_HASH,
        "family": "FE", "topic": topic, "arch": arch,
        "learning_rate": lr, "weight_decay": weight_decay,
        "master_seed": master_seed, "derived_seeds": seeds,
        "run_id": run_id, "best_epoch": best_epoch, "epochs_ran": len(history),
        "best_val_weighted_macro_f1": best_metrics["weighted_macro_f1"],
        "best_val_weighted_accuracy": best_metrics["weighted_accuracy"],
        "best_val_weighted_cross_entropy": best_metrics["weighted_cross_entropy"],
        "best_val_raw_macro_f1": best_metrics["raw_macro_f1"],
        "runtime_seconds": elapsed,
        **run_grad_stats,
        "gradient_clipping_max_norm": MAX_GRAD_NORM,
        "history_path": str(hist_path.relative_to(PROJECT_ROOT)),
        "checkpoint_path": str(checkpoint_path.relative_to(PROJECT_ROOT)) if checkpoint_path else None,
        "test_evaluated": False,
    }
    atomic_json_dump(rec, record_path)
    del model, optimizer, best_state
    cuda_cleanup()
    return rec


## FE Stage 1

All three architectures are searched with the predefined FE learning-rate and weight-decay grid.

In [10]:
# 10) FE Stage 1
FE_STAGE1_DIR = OUTPUT_DIR / "fe_stage1_search"
fe_stage1_grid = []
order = 0
for arch in ARCHITECTURES:
    for lr in FE_LEARNING_RATES:
        for wd in FE_WEIGHT_DECAYS:
            fe_stage1_grid.append({"arch": arch, "learning_rate": lr, "weight_decay": wd, "config_order": order})
            order += 1

fe1_records = []
for topic in TOPICS:
    for cfg in fe_stage1_grid:
        run_id = f"fe1_{topic}_{cfg['arch']}_lr{cfg['learning_rate']:.0e}_wd{cfg['weight_decay']:.0e}"
        print("FE1", topic, cfg, run_id)
        rec = train_frozen_run(topic, cfg["arch"], cfg["learning_rate"], cfg["weight_decay"], SEARCH_MASTER_SEED, run_id, FE_STAGE1_DIR, False)
        rec = {**rec, "config_order": cfg["config_order"]}
        fe1_records.append(rec)

fe1_df = pd.DataFrame(fe1_records)
atomic_csv_dump(fe1_df, FE_STAGE1_DIR / "all_results.csv")

# Top 2 inside each architecture/topic.
fe1_candidates = []
for topic in TOPICS:
    for arch in ARCHITECTURES:
        sub = fe1_df[(fe1_df.topic == topic) & (fe1_df.arch == arch)].copy()
        sub = sub.sort_values(
            ["best_val_weighted_macro_f1", "best_val_weighted_accuracy", "best_val_weighted_cross_entropy", "config_order"],
            ascending=[False, False, True, True], kind="mergesort"
        ).head(FE_KEEP_PER_ARCH)
        for r in sub.to_dict("records"):
            r["config_id"] = f"{arch}_lr{r['learning_rate']:.0e}_wd{r['weight_decay']:.0e}"
            fe1_candidates.append(r)
fe1_candidates_df = pd.DataFrame(fe1_candidates)
atomic_csv_dump(fe1_candidates_df, FE_STAGE1_DIR / "top2_per_architecture.csv")
display(fe1_candidates_df[["topic","arch","learning_rate","weight_decay","best_val_weighted_macro_f1","best_epoch"]])

for topic in TOPICS:
    sub = fe1_df[fe1_df.topic == topic].copy().sort_values("best_val_weighted_macro_f1", ascending=False)
    fig, ax = plt.subplots(figsize=(10, 6))
    labels = [f"{r.arch} | {r.learning_rate:.0e} | wd={r.weight_decay:g}" for r in sub.itertuples()]
    ax.barh(range(len(sub)), sub.best_val_weighted_macro_f1)
    ax.set_yticks(range(len(sub)), labels)
    ax.invert_yaxis(); ax.set_xlabel("Best validation weighted macro-F1")
    ax.set_title(f"FE Stage 1 — {topic}")
    save_figure(fig, FE_STAGE1_DIR / "plots" / f"{topic}_all_configs.png")


FE1 topicA {'arch': '512', 'learning_rate': 0.0003, 'weight_decay': 0.0, 'config_order': 0} fe1_topicA_512_lr3e-04_wd0e+00
FE1 topicA {'arch': '512', 'learning_rate': 0.0003, 'weight_decay': 0.01, 'config_order': 1} fe1_topicA_512_lr3e-04_wd1e-02
FE1 topicA {'arch': '512', 'learning_rate': 0.001, 'weight_decay': 0.0, 'config_order': 2} fe1_topicA_512_lr1e-03_wd0e+00
FE1 topicA {'arch': '512', 'learning_rate': 0.001, 'weight_decay': 0.01, 'config_order': 3} fe1_topicA_512_lr1e-03_wd1e-02
FE1 topicA {'arch': '512', 'learning_rate': 0.003, 'weight_decay': 0.0, 'config_order': 4} fe1_topicA_512_lr3e-03_wd0e+00
FE1 topicA {'arch': '512', 'learning_rate': 0.003, 'weight_decay': 0.01, 'config_order': 5} fe1_topicA_512_lr3e-03_wd1e-02
FE1 topicA {'arch': '512', 'learning_rate': 0.01, 'weight_decay': 0.0, 'config_order': 6} fe1_topicA_512_lr1e-02_wd0e+00
FE1 topicA {'arch': '512', 'learning_rate': 0.01, 'weight_decay': 0.01, 'config_order': 7} fe1_topicA_512_lr1e-02_wd1e-02
FE1 topicA {'arch': 

,topic,arch,learning_rate,weight_decay,best_val_weighted_macro_f1,best_epoch
0,topicA,512,0.003,0.00,0.738839,55
1,topicA,512,0.003,0.01,0.738839,55
2,topicA,MEAN,0.010,0.00,0.805805,53
3,topicA,MEAN,0.010,0.01,0.799586,53
4,topicA,ATTN,0.010,0.00,0.741890,39
5,topicA,ATTN,0.010,0.01,0.695788,17
6,topicB,512,0.010,0.00,0.742436,36
7,topicB,512,0.010,0.01,0.742074,36
8,topicB,MEAN,0.010,0.01,0.781511,48
9,topicB,MEAN,0.010,0.00,0.760336,35


## FE Stage 2

The strongest two settings per architecture are confirmed over five seeds, and the final FE setting is selected from the seed-average validation results.

In [11]:
# 11) FE Stage 2 confirmation and selected frozen models
FE_STAGE2_DIR = OUTPUT_DIR / "fe_stage2_confirmation"
fe2_records = []
for row in fe1_candidates_df.itertuples(index=False):
    for master_seed in CONFIRMATION_MASTER_SEEDS:
        run_id = f"fe2_{row.topic}_{row.config_id}_seed{master_seed}"
        print("FE2", run_id)
        rec = train_frozen_run(row.topic, row.arch, float(row.learning_rate), float(row.weight_decay), master_seed, run_id, FE_STAGE2_DIR, True)
        rec = {**rec, "config_id": row.config_id, "config_order": int(row.config_order)}
        fe2_records.append(rec)

fe2_df = pd.DataFrame(fe2_records)
atomic_csv_dump(fe2_df, FE_STAGE2_DIR / "all_seed_runs.csv")
fe2_summary = (
    fe2_df.groupby(["topic","arch","config_id","learning_rate","weight_decay","config_order"], as_index=False)
    .agg(
        mean_best_val_weighted_macro_f1=("best_val_weighted_macro_f1","mean"),
        std_best_val_weighted_macro_f1=("best_val_weighted_macro_f1","std"),
        mean_best_val_weighted_accuracy=("best_val_weighted_accuracy","mean"),
        mean_best_val_weighted_cross_entropy=("best_val_weighted_cross_entropy","mean"),
        mean_best_epoch=("best_epoch","mean"),
    )
)
atomic_csv_dump(fe2_summary, FE_STAGE2_DIR / "config_summary.csv")

FE_SELECTED = {}
selected_rows = []
for topic in TOPICS:
    FE_SELECTED[topic] = {}
    for arch in ARCHITECTURES:
        rows = fe2_summary[(fe2_summary.topic == topic) & (fe2_summary.arch == arch)].to_dict("records")
        winner = sorted(rows, key=confirmation_sort_key)[0]
        FE_SELECTED[topic][arch] = winner
        selected_rows.append(winner)
        # Copy tiny selected head checkpoints/predictions to stable selected-model paths.
        final_dir = OUTPUT_DIR / "selected_models" / topic / f"FE-{arch}"
        final_dir.mkdir(parents=True, exist_ok=True)
        for seed in CONFIRMATION_MASTER_SEEDS:
            rr = fe2_df[(fe2_df.topic == topic) & (fe2_df.arch == arch) & (fe2_df.config_id == winner["config_id"]) & (fe2_df.master_seed == seed)].iloc[0]
            src = PROJECT_ROOT / rr.checkpoint_path
            shutil.copy2(src, final_dir / f"head_seed_{seed}.pt")
        atomic_json_dump({"family":"FE", "topic":topic, "arch":arch, "selected_config":winner, "confirmation_seeds":CONFIRMATION_MASTER_SEEDS}, final_dir / "metadata.json")

atomic_json_dump(FE_SELECTED, FE_STAGE2_DIR / "selected_configs.json")
fe_selected_df = pd.DataFrame(selected_rows)
atomic_csv_dump(fe_selected_df, FE_STAGE2_DIR / "selected_configs.csv")
display(fe_selected_df[["topic","arch","learning_rate","weight_decay","mean_best_val_weighted_macro_f1","std_best_val_weighted_macro_f1"]])


FE2 fe2_topicA_512_lr3e-03_wd0e+00_seed20260912
FE2 fe2_topicA_512_lr3e-03_wd0e+00_seed20260918
FE2 fe2_topicA_512_lr3e-03_wd0e+00_seed20260924
FE2 fe2_topicA_512_lr3e-03_wd0e+00_seed20260930
FE2 fe2_topicA_512_lr3e-03_wd0e+00_seed20261006
FE2 fe2_topicA_512_lr3e-03_wd1e-02_seed20260912
FE2 fe2_topicA_512_lr3e-03_wd1e-02_seed20260918
FE2 fe2_topicA_512_lr3e-03_wd1e-02_seed20260924
FE2 fe2_topicA_512_lr3e-03_wd1e-02_seed20260930
FE2 fe2_topicA_512_lr3e-03_wd1e-02_seed20261006
FE2 fe2_topicA_MEAN_lr1e-02_wd0e+00_seed20260912
FE2 fe2_topicA_MEAN_lr1e-02_wd0e+00_seed20260918
FE2 fe2_topicA_MEAN_lr1e-02_wd0e+00_seed20260924
FE2 fe2_topicA_MEAN_lr1e-02_wd0e+00_seed20260930
FE2 fe2_topicA_MEAN_lr1e-02_wd0e+00_seed20261006
FE2 fe2_topicA_MEAN_lr1e-02_wd1e-02_seed20260912
FE2 fe2_topicA_MEAN_lr1e-02_wd1e-02_seed20260918
FE2 fe2_topicA_MEAN_lr1e-02_wd1e-02_seed20260924
FE2 fe2_topicA_MEAN_lr1e-02_wd1e-02_seed20260930
FE2 fe2_topicA_MEAN_lr1e-02_wd1e-02_seed20261006
FE2 fe2_topicA_ATTN_lr1e-02_wd

,topic,arch,learning_rate,weight_decay,mean_best_val_weighted_macro_f1,std_best_val_weighted_macro_f1
0,topicA,512,0.003,0.00,0.696092,0.031050
1,topicA,MEAN,0.010,0.00,0.778657,0.027964
2,topicA,ATTN,0.010,0.00,0.708320,0.010358
3,topicB,512,0.010,0.01,0.741786,0.012181
4,topicB,MEAN,0.010,0.00,0.768783,0.013855
5,topicB,ATTN,0.010,0.01,0.707366,0.019773


## Full fine-tuning

FT starts from the pretrained RoBERTa weights and updates the whole encoder together with the article-level head. Training uses AdamW, gradient clipping, mixed precision, warmup and linear learning-rate decay.

In [12]:
# 12) Full fine-tuning inference, evaluation and training

def encode_sequences_eval(model: FineTuneModel, seqs: Sequence[Sequence[int]], batch_size: int = INFERENCE_BATCH_SIZE) -> np.ndarray:
    parts = []
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(seqs), batch_size):
            ids, mask = collate_token_sequences(seqs[start:start+batch_size])
            parts.append(model.encode_chunks(ids, mask).cpu().numpy())
    return np.concatenate(parts, axis=0) if parts else np.empty((0, int(MODEL_SOURCE["hidden_size"])), dtype=np.float32)


def evaluate_ft(model: FineTuneModel, arch: str, df: pd.DataFrame):
    aids = df.article_id.tolist()
    attention_rows = []
    model.eval()
    if arch == "512":
        logits_parts = []
        with torch.inference_mode():
            for start in range(0, len(aids), INFERENCE_BATCH_SIZE):
                ba = aids[start:start + INFERENCE_BATCH_SIZE]
                seqs = [ARTICLE_CHUNKS[a]["chunks"][0] for a in ba]
                ids, mask = collate_token_sequences(seqs)
                h = model.encode_chunks(ids, mask)
                logits_parts.append(model.article_model.classify_article_embeddings(h).cpu().numpy())
        logits = np.concatenate(logits_parts, axis=0)
    else:
        # Efficiently encode every chunk in flat batches, then aggregate article vectors in moderate batches.
        flat_seqs = []
        offsets = [0]
        flat_counts = []
        for a in aids:
            flat_seqs.extend(ARTICLE_CHUNKS[a]["chunks"])
            flat_counts.extend(ARTICLE_CHUNKS[a]["content_counts"])
            offsets.append(len(flat_seqs))
        flat_h = encode_sequences_eval(model, flat_seqs, INFERENCE_BATCH_SIZE)
        logits_parts = []
        article_batch = 128
        with torch.inference_mode():
            for start in range(0, len(aids), article_batch):
                ba = aids[start:start + article_batch]
                idx0 = start
                chunks = [flat_h[offsets[i]:offsets[i+1]] for i in range(idx0, min(idx0+article_batch, len(aids)))]
                counts = [np.asarray(flat_counts[offsets[i]:offsets[i+1]], dtype=np.float32) for i in range(idx0, min(idx0+article_batch, len(aids)))]
                max_m = max(len(x) for x in chunks); H = flat_h.shape[1]
                h_pad = np.zeros((len(chunks), max_m, H), dtype=np.float32)
                c_pad = np.zeros((len(chunks), max_m), dtype=np.float32)
                m_pad = np.zeros((len(chunks), max_m), dtype=bool)
                for i,(hh,cc) in enumerate(zip(chunks,counts)):
                    m=len(hh); h_pad[i,:m]=hh; c_pad[i,:m]=cc; m_pad[i,:m]=True
                ht=torch.from_numpy(h_pad).to(DEVICE); ct=torch.from_numpy(c_pad).to(DEVICE); mt=torch.from_numpy(m_pad).to(DEVICE)
                article_h, weights = model.article_model.aggregate(ht, ct, mt)
                logits_parts.append(model.article_model.classify_article_embeddings(article_h).cpu().numpy())
                if arch == "ATTN":
                    for i, aid in enumerate(ba):
                        n = len(chunks[i])
                        ww = weights[i,:n].float().cpu().numpy()
                        for j,(w,c) in enumerate(zip(ww,counts[i])):
                            attention_rows.append({"article_id":aid,"chunk_index":j,"content_tokens":int(c),"attention_weight":float(w)})
        logits = np.concatenate(logits_parts, axis=0)
    y = df.label_id.to_numpy(np.int64)
    weights = df.balance_weight.to_numpy(np.float64)
    return logits, evaluate_logits(logits, y, weights), attention_rows


def optimizer_step_with_scaler(model, optimizer, scaler, scheduler=None) -> Tuple[bool, float]:
    scaler.unscale_(optimizer)

    # Returns the global norm BEFORE clipping.
    grad_norm_t = torch.nn.utils.clip_grad_norm_(
        model.parameters(), MAX_GRAD_NORM
    )
    grad_norm = float(grad_norm_t.detach().cpu())

    scale_before = float(scaler.get_scale())

    # GradScaler automatically skips optimizer.step() if unscale_ detected
    # inf/NaN gradients, then reduces the scale in update().
    scaler.step(optimizer)
    scaler.update()

    scale_after = float(scaler.get_scale())
    stepped = scale_after >= scale_before

    # A non-finite norm is acceptable only when AMP actually skipped the step.
    if stepped and not math.isfinite(grad_norm):
        raise FloatingPointError(
            f"Non-finite gradient norm ({grad_norm}) but GradScaler did not skip the optimizer step."
        )

    if stepped and scheduler is not None:
        scheduler.step()

    return stepped, grad_norm


def train_ft512_logical_batch(model, aids, y_by_id, optimizer, scaler, physical_batch_size) -> float:
    optimizer.zero_grad(set_to_none=True)
    total = len(aids); loss_sum = 0.0
    for start in range(0, total, physical_batch_size):
        ba = aids[start:start+physical_batch_size]
        seqs = [ARTICLE_CHUNKS[a]["chunks"][0] for a in ba]
        ids, mask = collate_token_sequences(seqs)
        h = model.encode_chunks(ids, mask)
        logits = model.article_model.classify_article_embeddings(h)
        y = torch.tensor([y_by_id[a] for a in ba], dtype=torch.long, device=DEVICE)
        loss = F.cross_entropy(logits, y)
        if not torch.isfinite(loss): raise FloatingPointError("Non-finite FT512 loss")
        # Exact logical-batch mean despite hardware microbatching.
        scaler.scale(loss * (len(ba) / total)).backward()
        loss_sum += float(loss.detach().cpu()) * len(ba)
    return loss_sum / total


def train_hier_logical_batch(model, aids, y_by_id, optimizer, scaler, chunk_microbatch) -> float:
    optimizer.zero_grad(set_to_none=True)
    total = len(aids); loss_sum = 0.0
    for aid in aids:
        if HIER_TRAIN_METHOD == "direct":
            loss, _ = direct_hier_article_backward(model, aid, int(y_by_id[aid]), 1.0 / total, scaler, chunk_microbatch)
        elif HIER_TRAIN_METHOD == "replay":
            loss, _ = cached_hier_article_backward(model, aid, int(y_by_id[aid]), 1.0 / total, scaler, chunk_microbatch)
        else:
            raise ValueError(HIER_TRAIN_METHOD)
        loss_sum += loss
    return loss_sum / total


def ft_checkpoint_payload(model: FineTuneModel, rec_stub: Mapping[str, Any]) -> Dict[str, Any]:
    return {"state_dict": cpu_state_dict(model), "metadata": dict(rec_stub)}


def save_ft_checkpoint_atomic(model: FineTuneModel, path: Path, metadata: Mapping[str, Any]) -> None:
    payload = ft_checkpoint_payload(model, metadata)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp)
    tmp.replace(path)


def train_ft_run(
    topic: str, arch: str, lr: float, weight_decay: float, master_seed: int,
    run_id: str, out_dir: Path, save_checkpoint: bool,
) -> Dict[str, Any]:
    record_path = out_dir / "run_records" / f"{run_id}.json"
    checkpoint_path = out_dir / "checkpoints" / f"{run_id}.pt" if save_checkpoint else None
    existing = load_run_record_if_complete(
        record_path, checkpoint_required=save_checkpoint, expected_protocol_hash=FT_PROTOCOL_HASH
    )
    if existing is not None:
        return existing

    train_df = TOPIC_DATA[topic]["train"]; val_df = TOPIC_DATA[topic]["validation"]
    y_by_id = train_df.set_index("article_id").label_id.to_dict()
    sampler, seeds = make_sampler(master_seed, topic)
    _use_gc = FT512_USE_GRADIENT_CHECKPOINTING if arch == "512" else HIER_USE_GRADIENT_CHECKPOINTING
    model = FineTuneModel(arch, seeds, gradient_checkpointing=_use_gc).to(DEVICE)
    set_training_rng(seeds["torch_seed"])
    optimizer = make_optimizer(model, lr, weight_decay)
    steps_per_epoch = math.ceil(sampler.expected_epoch_size(topic) / EFFECTIVE_ARTICLE_BATCH_SIZE)
    total_steps = steps_per_epoch * FT_MAX_EPOCHS
    warmup_steps = int(math.ceil(total_steps * FT_WARMUP_RATIO))
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )
    scaler = make_grad_scaler(USE_AMP)

    # Pure instrumentation: this file is overwritten at every major phase so that,
    # if a long CUDA/native call stalls, the last completed phase is visible on disk.
    runtime_status_dir = out_dir / "runtime_status"
    runtime_status_path = runtime_status_dir / f"{run_id}.json"

    def _write_runtime_status(phase: str, epoch_number: int | None = None, **extra) -> None:
        payload = {
            "run_id": run_id,
            "topic": topic,
            "arch": arch,
            "learning_rate": float(lr),
            "weight_decay": float(weight_decay),
            "master_seed": int(master_seed),
            "phase": phase,
            "epoch": int(epoch_number) if epoch_number is not None else None,
            "wall_time_unix": float(time.time()),
            "protocol_hash": FT_PROTOCOL_HASH,
            **extra,
        }
        atomic_json_dump(payload, runtime_status_path)

    history=[]; best_metrics=None; best_epoch=None; best_state=None
    meaningful_anchor=-np.inf; epochs_since_meaningful=0; skipped_steps=0
    all_grad_norms: List[float] = []
    t0=time.time()
    _write_runtime_status("run_start")

    for epoch in range(FT_MAX_EPOCHS):
        epoch_number = epoch + 1
        epoch_t0 = time.perf_counter()

        # ---------------- Training phase ----------------
        if VERBOSE_EPOCHS:
            print(f"{run_id} epoch {epoch_number} TRAIN start", flush=True)
        _write_runtime_status("train", epoch_number)

        train_phase_t0 = time.perf_counter()
        model.train()
        ordered_ids = sampler.article_ids_for_epoch(epoch, topic)
        sampled_loss_sum=0.0; sampled_n=0
        epoch_grad_norms: List[float] = []
        epoch_skipped_steps = 0

        for aids in logical_batches(ordered_ids):
            if arch == "512":
                batch_loss = train_ft512_logical_batch(
                    model, aids, y_by_id, optimizer, scaler, FT512_PHYSICAL_BATCH_SIZE
                )
            else:
                batch_loss = train_hier_logical_batch(
                    model, aids, y_by_id, optimizer, scaler, HIER_CHUNK_MICROBATCH_SIZE
                )
            stepped, grad_norm = optimizer_step_with_scaler(model, optimizer, scaler, scheduler)

            if stepped:
                epoch_grad_norms.append(grad_norm)
                all_grad_norms.append(grad_norm)
            else:
                skipped_steps += 1
                epoch_skipped_steps += 1
            sampled_loss_sum += batch_loss * len(aids); sampled_n += len(aids)

        train_phase_seconds = time.perf_counter() - train_phase_t0
        if VERBOSE_EPOCHS:
            print(
                f"{run_id} epoch {epoch_number} TRAIN done "
                f"({train_phase_seconds:.1f}s; amp_skips={epoch_skipped_steps})",
                flush=True,
            )

        # ---------------- Full-train evaluation phase ----------------
        if VERBOSE_EPOCHS:
            print(f"{run_id} epoch {epoch_number} TRAIN_EVAL start", flush=True)
        _write_runtime_status(
            "train_eval", epoch_number, train_phase_seconds=float(train_phase_seconds)
        )

        train_eval_t0 = time.perf_counter()
        train_logits, train_eval, _ = evaluate_ft(model, arch, train_df)
        train_eval_seconds = time.perf_counter() - train_eval_t0

        if VERBOSE_EPOCHS:
            print(
                f"{run_id} epoch {epoch_number} TRAIN_EVAL done "
                f"({train_eval_seconds:.1f}s)",
                flush=True,
            )

        # ---------------- Validation evaluation phase ----------------
        if VERBOSE_EPOCHS:
            print(f"{run_id} epoch {epoch_number} VAL_EVAL start", flush=True)
        _write_runtime_status(
            "val_eval",
            epoch_number,
            train_phase_seconds=float(train_phase_seconds),
            train_eval_seconds=float(train_eval_seconds),
        )

        val_eval_t0 = time.perf_counter()
        val_logits, val_eval, _ = evaluate_ft(model, arch, val_df)
        val_eval_seconds = time.perf_counter() - val_eval_t0
        epoch_seconds = time.perf_counter() - epoch_t0

        if VERBOSE_EPOCHS:
            print(
                f"{run_id} epoch {epoch_number} VAL_EVAL done "
                f"({val_eval_seconds:.1f}s)",
                flush=True,
            )

        epoch_grad_stats = summarize_preclip_grad_norms(epoch_grad_norms)
        row={
            "epoch":epoch_number,
            "sampled_train_loss":sampled_loss_sum/sampled_n,
            "full_train_weighted_macro_f1":train_eval["weighted_macro_f1"],
            "full_train_weighted_accuracy":train_eval["weighted_accuracy"],
            "full_train_weighted_cross_entropy":train_eval["weighted_cross_entropy"],
            "full_train_raw_macro_f1":train_eval["raw_macro_f1"],
            "val_weighted_macro_f1":val_eval["weighted_macro_f1"],
            "val_weighted_accuracy":val_eval["weighted_accuracy"],
            "val_weighted_cross_entropy":val_eval["weighted_cross_entropy"],
            "val_raw_macro_f1":val_eval["raw_macro_f1"],
            "val_raw_accuracy":val_eval["raw_accuracy"],
            "lr_after_epoch":float(optimizer.param_groups[0]["lr"]),
            "amp_skipped_optimizer_steps_epoch":int(epoch_skipped_steps),
            "train_phase_seconds":float(train_phase_seconds),
            "train_eval_seconds":float(train_eval_seconds),
            "val_eval_seconds":float(val_eval_seconds),
            "epoch_seconds":float(epoch_seconds),
            **epoch_grad_stats,
        }
        history.append(row)

        # Persist the completed-epoch history incrementally. This is diagnostic only;
        # an interrupted run still restarts from epoch 1 under the frozen protocol.
        partial_hist_path = out_dir / "histories_partial" / f"{run_id}.csv"
        atomic_csv_dump(pd.DataFrame(history), partial_hist_path)

        if checkpoint_is_better(val_eval,best_metrics):
            best_metrics=copy.deepcopy(val_eval); best_epoch=epoch_number
            if save_checkpoint:
                best_state=cpu_state_dict(model)

        current=val_eval["weighted_macro_f1"]
        if current > meaningful_anchor + EARLY_STOPPING_MIN_DELTA:
            meaningful_anchor=current; epochs_since_meaningful=0
        else:
            epochs_since_meaningful += 1

        _write_runtime_status(
            "epoch_complete",
            epoch_number,
            train_phase_seconds=float(train_phase_seconds),
            train_eval_seconds=float(train_eval_seconds),
            val_eval_seconds=float(val_eval_seconds),
            epoch_seconds=float(epoch_seconds),
            val_weighted_macro_f1=float(current),
            best_val_weighted_macro_f1=float(best_metrics["weighted_macro_f1"]),
            patience_count=int(epochs_since_meaningful),
            patience_limit=int(FT_PATIENCE),
            amp_skipped_optimizer_steps_epoch=int(epoch_skipped_steps),
        )

        if VERBOSE_EPOCHS:
            print(
                run_id, "epoch", epoch_number,
                "val_wF1", round(current,4),
                "best_wF1", round(best_metrics["weighted_macro_f1"],4),
                "patience", f"{epochs_since_meaningful}/{FT_PATIENCE}",
                "amp_skips", epoch_skipped_steps,
                "phase_s",
                f"train={train_phase_seconds:.1f}",
                f"train_eval={train_eval_seconds:.1f}",
                f"val_eval={val_eval_seconds:.1f}",
                f"total={epoch_seconds:.1f}",
                "lr", optimizer.param_groups[0]["lr"],
                flush=True,
            )

        if epoch_number >= MIN_EPOCHS_BY_TOPIC[topic] and epochs_since_meaningful >= FT_PATIENCE:
            break

    hist_path=out_dir/"histories"/f"{run_id}.csv"
    atomic_csv_dump(pd.DataFrame(history),hist_path)
    pred_path=None; attn_path=None

    if save_checkpoint:
        assert best_state is not None
        if VERBOSE_EPOCHS:
            print(f"{run_id} FINALIZE checkpoint/predictions start", flush=True)
        _write_runtime_status("finalize", best_epoch)

        model.load_state_dict(best_state)
        meta_stub={
            "topic":topic,"arch":arch,"learning_rate":lr,"weight_decay":weight_decay,
            "master_seed":master_seed,"best_epoch":best_epoch,
            "protocol_hash":FT_PROTOCOL_HASH,"base_protocol_hash":PROTOCOL_HASH
        }
        # Save the already-captured CPU best state directly (avoid a second ~500 MB GPU→CPU copy).
        checkpoint_path.parent.mkdir(parents=True,exist_ok=True)
        _tmp_cp = checkpoint_path.with_suffix(checkpoint_path.suffix + ".tmp")
        torch.save({"state_dict": best_state, "metadata": meta_stub}, _tmp_cp)
        _tmp_cp.replace(checkpoint_path)
        val_logits, val_check, attn_rows=evaluate_ft(model,arch,val_df)
        assert abs(val_check["weighted_macro_f1"]-best_metrics["weighted_macro_f1"])<1e-10
        pred_path=out_dir/"validation_predictions"/f"{run_id}.csv"
        save_predictions(val_df,val_logits,pred_path)
        if arch=="ATTN":
            attn_path=out_dir/"attention_weights"/f"{run_id}.csv"
            atomic_csv_dump(pd.DataFrame(attn_rows),attn_path)

        if VERBOSE_EPOCHS:
            print(f"{run_id} FINALIZE checkpoint/predictions done", flush=True)

    elapsed=time.time()-t0
    run_grad_stats = summarize_preclip_grad_norms(all_grad_norms)
    hist_df = pd.DataFrame(history)

    rec={
        "protocol_hash":FT_PROTOCOL_HASH,"base_protocol_hash":PROTOCOL_HASH,"family":"FT","topic":topic,"arch":arch,
        "learning_rate":lr,"weight_decay":weight_decay,"master_seed":master_seed,"derived_seeds":seeds,
        "run_id":run_id,"best_epoch":best_epoch,"epochs_ran":len(history),
        "best_val_weighted_macro_f1":best_metrics["weighted_macro_f1"],
        "best_val_weighted_accuracy":best_metrics["weighted_accuracy"],
        "best_val_weighted_cross_entropy":best_metrics["weighted_cross_entropy"],
        "best_val_raw_macro_f1":best_metrics["raw_macro_f1"],
        "runtime_seconds":elapsed,"history_path":str(hist_path.relative_to(PROJECT_ROOT)),
        "checkpoint_path":str(checkpoint_path.relative_to(PROJECT_ROOT)) if checkpoint_path else None,
        "validation_predictions_path":str(pred_path.relative_to(PROJECT_ROOT)) if pred_path else None,
        "attention_weights_path":str(attn_path.relative_to(PROJECT_ROOT)) if attn_path else None,
        "steps_per_epoch":steps_per_epoch,"max_scheduler_steps":total_steps,"warmup_steps":warmup_steps,
        "ft_max_epochs":FT_MAX_EPOCHS,"ft_patience":FT_PATIENCE,
        "skipped_amp_optimizer_steps":skipped_steps,
        "mean_train_phase_seconds":float(hist_df.train_phase_seconds.mean()),
        "mean_train_eval_seconds":float(hist_df.train_eval_seconds.mean()),
        "mean_val_eval_seconds":float(hist_df.val_eval_seconds.mean()),
        "mean_epoch_seconds":float(hist_df.epoch_seconds.mean()),
        "max_epoch_seconds":float(hist_df.epoch_seconds.max()),
        **run_grad_stats,
        "gradient_clipping_max_norm":MAX_GRAD_NORM,
        "ft512_physical_batch_size":FT512_PHYSICAL_BATCH_SIZE if arch=="512" else None,
        "ft_gradient_checkpointing":bool(_use_gc),
        "hier_train_method":HIER_TRAIN_METHOD if arch!="512" else None,
        "hier_chunk_microbatch_size":HIER_CHUNK_MICROBATCH_SIZE if arch!="512" else None,
        "runtime_status_path":str(runtime_status_path.relative_to(PROJECT_ROOT)),
        "test_evaluated":False,
    }
    atomic_json_dump(rec,record_path)
    _write_runtime_status(
        "complete",
        len(history),
        runtime_seconds=float(elapsed),
        best_epoch=int(best_epoch),
        best_val_weighted_macro_f1=float(best_metrics["weighted_macro_f1"]),
    )

    del model,optimizer,scheduler,scaler,best_state
    cuda_cleanup()
    return rec



## FT Stage 1

Each architecture is evaluated with all six predefined learning-rate/weight-decay combinations for both topics.

In [13]:
# 13) FT Stage 1 — full search on 512, MEAN, and ATTN
FT_STAGE1_DIR = OUTPUT_DIR / "ft_stage1_full_search"
ft1_grid = []
order = 0
for arch in ARCHITECTURES:
    for lr in FT_LEARNING_RATES:
        for wd in FT_WEIGHT_DECAYS:
            config_id = f"{arch}_lr{lr:.0e}_wd{wd:.0e}"
            ft1_grid.append({
                "arch": arch,
                "learning_rate": lr,
                "weight_decay": wd,
                "config_order": order,
                "config_id": config_id,
            })
            order += 1

assert len(ft1_grid) == len(ARCHITECTURES) * len(FT_LEARNING_RATES) * len(FT_WEIGHT_DECAYS) == 18

ft1_records = []
for topic in TOPICS:
    for cfg in ft1_grid:
        run_id = f"ft1_{topic}_{cfg['config_id']}"
        print("FT1", run_id)
        rec = train_ft_run(
            topic, cfg["arch"], float(cfg["learning_rate"]), float(cfg["weight_decay"]),
            SEARCH_MASTER_SEED, run_id, FT_STAGE1_DIR, False,
        )
        ft1_records.append({**rec, "config_id": cfg["config_id"], "config_order": int(cfg["config_order"])})

ft1_df = pd.DataFrame(ft1_records)
assert len(ft1_df) == len(TOPICS) * len(ft1_grid) == 36
atomic_csv_dump(ft1_df, FT_STAGE1_DIR / "all_results.csv")

# Top 2 search configurations inside each topic/architecture.
ft1_candidates = []
for topic in TOPICS:
    for arch in ARCHITECTURES:
        sub = ft1_df[(ft1_df.topic == topic) & (ft1_df.arch == arch)].copy()
        sub = sub.sort_values(
            ["best_val_weighted_macro_f1", "best_val_weighted_accuracy", "best_val_weighted_cross_entropy", "config_order"],
            ascending=[False, False, True, True], kind="mergesort",
        ).head(FT_KEEP_PER_ARCH)
        assert len(sub) == FT_KEEP_PER_ARCH
        ft1_candidates.extend(sub.to_dict("records"))

ft1_candidates_df = pd.DataFrame(ft1_candidates)
assert (ft1_candidates_df.groupby(["topic", "arch"]).size() == FT_KEEP_PER_ARCH).all()
atomic_csv_dump(ft1_candidates_df, FT_STAGE1_DIR / "top2_per_architecture.csv")
display(ft1_candidates_df[["topic", "arch", "learning_rate", "weight_decay", "best_val_weighted_macro_f1", "best_val_weighted_accuracy"]])


FT1 ft1_topicA_512_lr1e-05_wd0e+00
FT1 ft1_topicA_512_lr1e-05_wd1e-02
FT1 ft1_topicA_512_lr2e-05_wd0e+00
FT1 ft1_topicA_512_lr2e-05_wd1e-02
FT1 ft1_topicA_512_lr3e-05_wd0e+00
FT1 ft1_topicA_512_lr3e-05_wd1e-02
FT1 ft1_topicA_MEAN_lr1e-05_wd0e+00
FT1 ft1_topicA_MEAN_lr1e-05_wd1e-02
FT1 ft1_topicA_MEAN_lr2e-05_wd0e+00
FT1 ft1_topicA_MEAN_lr2e-05_wd1e-02
FT1 ft1_topicA_MEAN_lr3e-05_wd0e+00
FT1 ft1_topicA_MEAN_lr3e-05_wd1e-02
FT1 ft1_topicA_ATTN_lr1e-05_wd0e+00
FT1 ft1_topicA_ATTN_lr1e-05_wd1e-02
FT1 ft1_topicA_ATTN_lr2e-05_wd0e+00
FT1 ft1_topicA_ATTN_lr2e-05_wd1e-02
FT1 ft1_topicA_ATTN_lr3e-05_wd0e+00
FT1 ft1_topicA_ATTN_lr3e-05_wd1e-02
FT1 ft1_topicB_512_lr1e-05_wd0e+00
FT1 ft1_topicB_512_lr1e-05_wd1e-02
FT1 ft1_topicB_512_lr2e-05_wd0e+00
FT1 ft1_topicB_512_lr2e-05_wd1e-02
FT1 ft1_topicB_512_lr3e-05_wd0e+00
FT1 ft1_topicB_512_lr3e-05_wd1e-02
FT1 ft1_topicB_MEAN_lr1e-05_wd0e+00
FT1 ft1_topicB_MEAN_lr1e-05_wd1e-02
FT1 ft1_topicB_MEAN_lr2e-05_wd0e+00
FT1 ft1_topicB_MEAN_lr2e-05_wd1e-02
FT1 

,topic,arch,learning_rate,weight_decay,best_val_weighted_macro_f1,best_val_weighted_accuracy
0,topicA,512,0.00001,0.01,0.950421,0.950269
1,topicA,512,0.00001,0.00,0.949211,0.948925
2,topicA,MEAN,0.00002,0.00,0.965426,0.965412
3,topicA,MEAN,0.00001,0.00,0.963242,0.963262
4,topicA,ATTN,0.00001,0.00,0.962279,0.962366
5,topicA,ATTN,0.00001,0.01,0.961445,0.961380
6,topicB,512,0.00002,0.01,0.955522,0.955501
7,topicB,512,0.00001,0.01,0.951208,0.951192
8,topicB,MEAN,0.00003,0.00,0.991420,0.991420
9,topicB,MEAN,0.00001,0.01,0.975471,0.975471


## FT Stage 2

The top two settings per topic and architecture are confirmed over three seeds. Final selection uses mean validation weighted macro-F1, followed by the predefined tie-break rules.

In [14]:
# 14) FT Stage 2 — three-seed confirmation and checkpoint retention
FT_STAGE2_DIR = OUTPUT_DIR / "ft_stage2_confirmation"

FT_SELECTED = {}
all_ft2_records = []
for topic in TOPICS:
    FT_SELECTED[topic] = {}
    for arch in ARCHITECTURES:
        final_dir = OUTPUT_DIR / "selected_models" / topic / f"FT-{arch}"
        selection_path = final_dir / "metadata.json"

        # Fully finalized architecture/topic can be reused directly on notebook resume.
        if selection_path.exists():
            meta = json_load(selection_path)
            if meta.get("ft_protocol_hash") == FT_PROTOCOL_HASH:
                expected_files = [final_dir / f"model_seed_{s}.pt" for s in FT_CONFIRMATION_MASTER_SEEDS]
                if all(p.exists() for p in expected_files):
                    FT_SELECTED[topic][arch] = meta["selected_config"]
                    for cfg in ft1_candidates_df[(ft1_candidates_df.topic == topic) & (ft1_candidates_df.arch == arch)].itertuples(index=False):
                        for seed in FT_CONFIRMATION_MASTER_SEEDS:
                            rp = FT_STAGE2_DIR / "run_records" / f"ft2_{topic}_{cfg.config_id}_seed{seed}.json"
                            if rp.exists():
                                all_ft2_records.append(json_load(rp))
                    print("FT2 already finalized:", topic, arch)
                    continue

        candidates = ft1_candidates_df[(ft1_candidates_df.topic == topic) & (ft1_candidates_df.arch == arch)].to_dict("records")
        assert len(candidates) == FT_KEEP_PER_ARCH
        arch_records = []
        for cfg in candidates:
            for seed in FT_CONFIRMATION_MASTER_SEEDS:
                run_id = f"ft2_{topic}_{cfg['config_id']}_seed{seed}"
                print("FT2", run_id)
                rec = train_ft_run(
                    topic, arch, float(cfg["learning_rate"]), float(cfg["weight_decay"]),
                    seed, run_id, FT_STAGE2_DIR, True,
                )
                rec = {**rec, "config_id": cfg["config_id"], "config_order": int(cfg["config_order"])}
                atomic_json_dump(rec, FT_STAGE2_DIR / "run_records" / f"{run_id}.json")
                arch_records.append(rec)
                all_ft2_records.append(rec)

        adf = pd.DataFrame(arch_records)
        summary = (
            adf.groupby(["config_id", "learning_rate", "weight_decay", "config_order"], as_index=False)
            .agg(
                mean_best_val_weighted_macro_f1=("best_val_weighted_macro_f1", "mean"),
                std_best_val_weighted_macro_f1=("best_val_weighted_macro_f1", "std"),
                mean_best_val_weighted_accuracy=("best_val_weighted_accuracy", "mean"),
                mean_best_val_weighted_cross_entropy=("best_val_weighted_cross_entropy", "mean"),
                mean_best_epoch=("best_epoch", "mean"),
                mean_skipped_amp_optimizer_steps=("skipped_amp_optimizer_steps", "mean"),
            )
        )
        winner = sorted(summary.to_dict("records"), key=confirmation_sort_key)[0]
        winner["topic"] = topic
        winner["arch"] = arch
        FT_SELECTED[topic][arch] = winner

        final_dir.mkdir(parents=True, exist_ok=True)
        for rec in arch_records:
            cp = PROJECT_ROOT / rec["checkpoint_path"]
            if rec["config_id"] == winner["config_id"]:
                dst = final_dir / f"model_seed_{rec['master_seed']}.pt"
                if dst.exists():
                    dst.unlink()
                cp.replace(dst)
                rec["checkpoint_path"] = str(dst.relative_to(PROJECT_ROOT))
                atomic_json_dump(rec, FT_STAGE2_DIR / "run_records" / f"{rec['run_id']}.json")
            else:
                if cp.exists():
                    cp.unlink()
                rec["checkpoint_path"] = None
                rec["checkpoint_deleted_after_candidate_selection"] = True
                atomic_json_dump(rec, FT_STAGE2_DIR / "run_records" / f"{rec['run_id']}.json")

        atomic_json_dump({
            "family": "FT",
            "topic": topic,
            "arch": arch,
            "base_protocol_hash": PROTOCOL_HASH,
            "ft_protocol_hash": FT_PROTOCOL_HASH,
            "selected_config": winner,
            "confirmation_seeds": FT_CONFIRMATION_MASTER_SEEDS,
        }, selection_path)
        atomic_csv_dump(summary, final_dir / "confirmation_candidate_summary.csv")

# Rebuild canonical Stage-2 tables from run records after any path updates/resume.
record_files = sorted((FT_STAGE2_DIR / "run_records").glob("*.json"))
ft2_all_df = pd.DataFrame([json_load(p) for p in record_files])
assert len(ft2_all_df) == len(TOPICS) * len(ARCHITECTURES) * FT_KEEP_PER_ARCH * len(FT_CONFIRMATION_MASTER_SEEDS)
atomic_csv_dump(ft2_all_df, FT_STAGE2_DIR / "all_seed_runs.csv")
ft2_summary = (
    ft2_all_df.groupby(["topic", "arch", "config_id", "learning_rate", "weight_decay", "config_order"], as_index=False)
    .agg(
        mean_best_val_weighted_macro_f1=("best_val_weighted_macro_f1", "mean"),
        std_best_val_weighted_macro_f1=("best_val_weighted_macro_f1", "std"),
        mean_best_val_weighted_accuracy=("best_val_weighted_accuracy", "mean"),
        mean_best_val_weighted_cross_entropy=("best_val_weighted_cross_entropy", "mean"),
        mean_best_epoch=("best_epoch", "mean"),
        mean_skipped_amp_optimizer_steps=("skipped_amp_optimizer_steps", "mean"),
    )
)
atomic_csv_dump(ft2_summary, FT_STAGE2_DIR / "config_summary.csv")
atomic_json_dump(FT_SELECTED, FT_STAGE2_DIR / "selected_configs.json")
print("FT selection complete")
display(ft2_summary.sort_values(["topic", "arch", "mean_best_val_weighted_macro_f1"], ascending=[True, True, False]))


FT2 already finalized: topicA 512
FT2 already finalized: topicA MEAN
FT2 already finalized: topicA ATTN
FT2 already finalized: topicB 512
FT2 already finalized: topicB MEAN
FT2 already finalized: topicB ATTN
FT selection complete


,topic,arch,config_id,learning_rate,weight_decay,config_order,mean_best_val_weighted_macro_f1,std_best_val_weighted_macro_f1,mean_best_val_weighted_accuracy,mean_best_val_weighted_cross_entropy,mean_best_epoch,mean_skipped_amp_optimizer_steps
1,topicA,512,512_lr1e-05_wd1e-02,0.00001,0.01,1,0.939886,0.006361,0.940024,0.236379,15.333333,5.000000
0,topicA,512,512_lr1e-05_wd0e+00,0.00001,0.00,0,0.930242,0.009193,0.930645,0.340003,16.000000,4.333333
3,topicA,ATTN,ATTN_lr1e-05_wd1e-02,0.00001,0.01,13,0.953912,0.004380,0.953943,0.196040,17.000000,3.666667
2,topicA,ATTN,ATTN_lr1e-05_wd0e+00,0.00001,0.00,12,0.951360,0.005093,0.951225,0.183221,13.666667,4.000000
4,topicA,MEAN,MEAN_lr1e-05_wd0e+00,0.00001,0.00,6,0.947256,0.019634,0.947342,0.170520,12.000000,3.666667
5,topicA,MEAN,MEAN_lr2e-05_wd0e+00,0.00002,0.00,8,0.947088,0.007866,0.947252,0.249640,12.000000,4.000000
7,topicB,512,512_lr2e-05_wd1e-02,0.00002,0.01,3,0.941312,0.015696,0.940994,0.360138,17.000000,4.333333
6,topicB,512,512_lr1e-05_wd1e-02,0.00001,0.01,1,0.927412,0.007998,0.926894,0.287072,17.333333,4.333333
9,topicB,ATTN,ATTN_lr3e-05_wd1e-02,0.00003,0.01,17,0.975331,0.013571,0.975347,0.105310,17.000000,3.666667
8,topicB,ATTN,ATTN_lr1e-05_wd0e+00,0.00001,0.00,12,0.975215,0.001948,0.975247,0.137816,23.000000,4.333333


## Random-initialized control

A small control uses the same RoBERTa architecture and tokenizer but starts the encoder from random weights. It uses one predefined training setting and does not affect FE or FT model selection.

In [15]:
# 15) Random-initialized RoBERTa control
#
from transformers import AutoConfig

RANDOM_INIT_CONTROL_DIR = OUTPUT_DIR / "random_init_control"
RANDOM_INIT_LEARNING_RATE = 1e-4
RANDOM_INIT_WEIGHT_DECAY = 0.01
RANDOM_INIT_MAX_EPOCHS = 100
RANDOM_INIT_WARMUP_RATIO = 0.10
RANDOM_INIT_MASTER_SEED = SEARCH_MASTER_SEED
RANDOM_INIT_EXPECTED_RUNS = len(TOPICS) * len(ARCHITECTURES)  # 2 * 3 = 6


def random_encoder_init_seed(master_seed: int, topic: str) -> int:
    # New stream only for the parameter initialization that pretrained FT does not have.
    # It depends on topic but not architecture, so 512/MEAN/ATTN start from the same
    # random RoBERTa encoder within a topic.
    return seed32(master_seed, "random-roberta-encoder-init", topic)


RANDOM_INIT_CONTROL_PROTOCOL_HASH = '9efe780c0f50e591a04e6c9e4212fd5e3b8f1cc52a456fd9d1def93127f966d1'


class RandomInitFineTuneModel(nn.Module):
    """Same article-level model interface as FineTuneModel, but with random RoBERTa encoder weights."""
    def __init__(
        self,
        arch: str,
        seeds: Mapping[str, int],
        encoder_init_seed: int,
        gradient_checkpointing: bool = FT_GRADIENT_CHECKPOINTING,
    ):
        super().__init__()
        self.arch = arch
        self.encoder_init_seed = int(encoder_init_seed)

        # AutoConfig comes from the pinned local RoBERTa assets, but encoder parameters are
        # created from scratch. Set the dedicated encoder-init RNG before construction.
        config = AutoConfig.from_pretrained(BASE_MODEL_DIR)
        set_training_rng(self.encoder_init_seed)
        try:
            self.encoder = AutoModel.from_config(config, attn_implementation=ATTN_IMPLEMENTATION)
        except (TypeError, ValueError):
            self.encoder = AutoModel.from_config(config)

        if gradient_checkpointing:
            try:
                self.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
            except TypeError:
                self.encoder.gradient_checkpointing_enable()
            if hasattr(self.encoder.config, "use_cache"):
                self.encoder.config.use_cache = False

        # Reuse the exact existing classifier/MEAN/ATTN definitions and their seeded initializers.
        self.article_model = ArticleHeadModel(arch, int(self.encoder.config.hidden_size), seeds)

    def encode_chunks(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        with autocast_encoder():
            out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
            h = out.last_hidden_state[:, 0, :]
        return h.float()


def train_random_init_run(topic: str, arch: str, master_seed: int = RANDOM_INIT_MASTER_SEED) -> Dict[str, Any]:
    run_id = f"ri_{topic}_{arch}_seed{master_seed}"
    record_path = RANDOM_INIT_CONTROL_DIR / "run_records" / f"{run_id}.json"
    selected_dir = OUTPUT_DIR / "selected_models" / topic / f"RANDOM-{arch}"
    checkpoint_path = selected_dir / f"model_seed_{master_seed}.pt"

    existing = load_run_record_if_complete(
        record_path, checkpoint_required=True, expected_protocol_hash=RANDOM_INIT_CONTROL_PROTOCOL_HASH
    )
    if existing is not None:
        return existing

    train_df = TOPIC_DATA[topic]["train"]
    val_df = TOPIC_DATA[topic]["validation"]
    y_by_id = train_df.set_index("article_id").label_id.to_dict()
    sampler, seeds = make_sampler(master_seed, topic)
    enc_seed = random_encoder_init_seed(master_seed, topic)
    _use_gc = FT512_USE_GRADIENT_CHECKPOINTING if arch == "512" else HIER_USE_GRADIENT_CHECKPOINTING

    # Encoder construction uses its own deterministic init stream. Reset the complete training
    # RNG immediately afterwards so Stage-1-compatible dropout/training randomness starts from
    # exactly the same derived stream as the existing FT search runs.
    model = RandomInitFineTuneModel(
        arch, seeds, encoder_init_seed=enc_seed, gradient_checkpointing=_use_gc
    ).to(DEVICE)
    set_training_rng(seeds["torch_seed"])

    optimizer = make_optimizer(model, RANDOM_INIT_LEARNING_RATE, RANDOM_INIT_WEIGHT_DECAY)
    steps_per_epoch = math.ceil(sampler.expected_epoch_size(topic) / EFFECTIVE_ARTICLE_BATCH_SIZE)
    total_steps = steps_per_epoch * RANDOM_INIT_MAX_EPOCHS
    warmup_steps = int(math.ceil(total_steps * RANDOM_INIT_WARMUP_RATIO))
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )
    scaler = make_grad_scaler(USE_AMP)

    runtime_status_path = RANDOM_INIT_CONTROL_DIR / "runtime_status" / f"{run_id}.json"

    def _write_runtime_status(phase: str, epoch_number: int | None = None, **extra) -> None:
        atomic_json_dump({
            "run_id": run_id,
            "topic": topic,
            "arch": arch,
            "learning_rate": RANDOM_INIT_LEARNING_RATE,
            "weight_decay": RANDOM_INIT_WEIGHT_DECAY,
            "master_seed": int(master_seed),
            "encoder_init_seed": int(enc_seed),
            "phase": phase,
            "epoch": int(epoch_number) if epoch_number is not None else None,
            "wall_time_unix": float(time.time()),
            "protocol_hash": RANDOM_INIT_CONTROL_PROTOCOL_HASH,
            **extra,
        }, runtime_status_path)

    history = []
    best_metrics = None
    best_epoch = None
    best_state = None
    skipped_steps = 0
    all_grad_norms: List[float] = []
    t0 = time.time()
    _write_runtime_status("run_start")

    # Intentionally fixed horizon: NO patience and NO early-stopping break.
    for epoch in range(RANDOM_INIT_MAX_EPOCHS):
        epoch_number = epoch + 1
        epoch_t0 = time.perf_counter()

        if VERBOSE_EPOCHS:
            print(f"{run_id} epoch {epoch_number}/{RANDOM_INIT_MAX_EPOCHS} TRAIN start", flush=True)
        _write_runtime_status("train", epoch_number)

        train_phase_t0 = time.perf_counter()
        model.train()
        ordered_ids = sampler.article_ids_for_epoch(epoch, topic)
        sampled_loss_sum = 0.0
        sampled_n = 0
        epoch_grad_norms: List[float] = []
        epoch_skipped_steps = 0

        for aids in logical_batches(ordered_ids):
            if arch == "512":
                batch_loss = train_ft512_logical_batch(
                    model, aids, y_by_id, optimizer, scaler, FT512_PHYSICAL_BATCH_SIZE
                )
            else:
                batch_loss = train_hier_logical_batch(
                    model, aids, y_by_id, optimizer, scaler, HIER_CHUNK_MICROBATCH_SIZE
                )
            stepped, grad_norm = optimizer_step_with_scaler(model, optimizer, scaler, scheduler)
            if stepped:
                epoch_grad_norms.append(grad_norm)
                all_grad_norms.append(grad_norm)
            else:
                skipped_steps += 1
                epoch_skipped_steps += 1
            sampled_loss_sum += batch_loss * len(aids)
            sampled_n += len(aids)

        train_phase_seconds = time.perf_counter() - train_phase_t0
        if VERBOSE_EPOCHS:
            print(f"{run_id} epoch {epoch_number} TRAIN done ({train_phase_seconds:.1f}s; amp_skips={epoch_skipped_steps})", flush=True)

        if VERBOSE_EPOCHS:
            print(f"{run_id} epoch {epoch_number} TRAIN_EVAL start", flush=True)
        _write_runtime_status("train_eval", epoch_number, train_phase_seconds=float(train_phase_seconds))
        train_eval_t0 = time.perf_counter()
        _, train_eval, _ = evaluate_ft(model, arch, train_df)
        train_eval_seconds = time.perf_counter() - train_eval_t0
        if VERBOSE_EPOCHS:
            print(f"{run_id} epoch {epoch_number} TRAIN_EVAL done ({train_eval_seconds:.1f}s)", flush=True)

        if VERBOSE_EPOCHS:
            print(f"{run_id} epoch {epoch_number} VAL_EVAL start", flush=True)
        _write_runtime_status(
            "val_eval", epoch_number,
            train_phase_seconds=float(train_phase_seconds),
            train_eval_seconds=float(train_eval_seconds),
        )
        val_eval_t0 = time.perf_counter()
        _, val_eval, _ = evaluate_ft(model, arch, val_df)
        val_eval_seconds = time.perf_counter() - val_eval_t0
        epoch_seconds = time.perf_counter() - epoch_t0
        if VERBOSE_EPOCHS:
            print(f"{run_id} epoch {epoch_number} VAL_EVAL done ({val_eval_seconds:.1f}s)", flush=True)

        epoch_grad_stats = summarize_preclip_grad_norms(epoch_grad_norms)
        row = {
            "epoch": epoch_number,
            "sampled_train_loss": sampled_loss_sum / sampled_n,
            "full_train_weighted_macro_f1": train_eval["weighted_macro_f1"],
            "full_train_weighted_accuracy": train_eval["weighted_accuracy"],
            "full_train_weighted_cross_entropy": train_eval["weighted_cross_entropy"],
            "full_train_raw_macro_f1": train_eval["raw_macro_f1"],
            "val_weighted_macro_f1": val_eval["weighted_macro_f1"],
            "val_weighted_accuracy": val_eval["weighted_accuracy"],
            "val_weighted_cross_entropy": val_eval["weighted_cross_entropy"],
            "val_raw_macro_f1": val_eval["raw_macro_f1"],
            "val_raw_accuracy": val_eval["raw_accuracy"],
            "lr_after_epoch": float(optimizer.param_groups[0]["lr"]),
            "amp_skipped_optimizer_steps_epoch": int(epoch_skipped_steps),
            "train_phase_seconds": float(train_phase_seconds),
            "train_eval_seconds": float(train_eval_seconds),
            "val_eval_seconds": float(val_eval_seconds),
            "epoch_seconds": float(epoch_seconds),
            **epoch_grad_stats,
        }
        history.append(row)
        atomic_csv_dump(
            pd.DataFrame(history), RANDOM_INIT_CONTROL_DIR / "histories_partial" / f"{run_id}.csv"
        )

        if checkpoint_is_better(val_eval, best_metrics):
            best_metrics = copy.deepcopy(val_eval)
            best_epoch = epoch_number
            best_state = cpu_state_dict(model)

        _write_runtime_status(
            "epoch_complete", epoch_number,
            train_phase_seconds=float(train_phase_seconds),
            train_eval_seconds=float(train_eval_seconds),
            val_eval_seconds=float(val_eval_seconds),
            epoch_seconds=float(epoch_seconds),
            val_weighted_macro_f1=float(val_eval["weighted_macro_f1"]),
            best_val_weighted_macro_f1=float(best_metrics["weighted_macro_f1"]),
            amp_skipped_optimizer_steps_epoch=int(epoch_skipped_steps),
            fixed_epoch_budget=int(RANDOM_INIT_MAX_EPOCHS),
        )

        if VERBOSE_EPOCHS:
            print(
                run_id, "epoch", f"{epoch_number}/{RANDOM_INIT_MAX_EPOCHS}",
                "val_wF1", round(val_eval["weighted_macro_f1"], 4),
                "best_wF1", round(best_metrics["weighted_macro_f1"], 4),
                "best_epoch", best_epoch,
                "amp_skips", epoch_skipped_steps,
                "phase_s",
                f"train={train_phase_seconds:.1f}",
                f"train_eval={train_eval_seconds:.1f}",
                f"val_eval={val_eval_seconds:.1f}",
                f"total={epoch_seconds:.1f}",
                "lr", optimizer.param_groups[0]["lr"],
                flush=True,
            )

    assert len(history) == RANDOM_INIT_MAX_EPOCHS
    assert best_state is not None and best_metrics is not None and best_epoch is not None
    hist_path = RANDOM_INIT_CONTROL_DIR / "histories" / f"{run_id}.csv"
    hist_df = pd.DataFrame(history)
    atomic_csv_dump(hist_df, hist_path)

    if VERBOSE_EPOCHS:
        print(f"{run_id} FINALIZE best checkpoint/predictions start", flush=True)
    _write_runtime_status("finalize", best_epoch)
    model.load_state_dict(best_state)

    selected_dir.mkdir(parents=True, exist_ok=True)
    meta_stub = {
        "family": "RANDOM_INIT",
        "topic": topic,
        "arch": arch,
        "learning_rate": RANDOM_INIT_LEARNING_RATE,
        "weight_decay": RANDOM_INIT_WEIGHT_DECAY,
        "master_seed": int(master_seed),
        "encoder_init_seed": int(enc_seed),
        "best_epoch": int(best_epoch),
        "protocol_hash": RANDOM_INIT_CONTROL_PROTOCOL_HASH,
        "base_protocol_hash": PROTOCOL_HASH,
        "ft_protocol_hash_reference": FT_PROTOCOL_HASH,
    }
    tmp_cp = checkpoint_path.with_suffix(checkpoint_path.suffix + ".tmp")
    torch.save({"state_dict": best_state, "metadata": meta_stub}, tmp_cp)
    tmp_cp.replace(checkpoint_path)

    val_logits, val_check, attn_rows = evaluate_ft(model, arch, val_df)
    assert abs(val_check["weighted_macro_f1"] - best_metrics["weighted_macro_f1"]) < 1e-10
    pred_path = RANDOM_INIT_CONTROL_DIR / "validation_predictions" / f"{run_id}.csv"
    save_predictions(val_df, val_logits, pred_path)
    attn_path = None
    if arch == "ATTN":
        attn_path = RANDOM_INIT_CONTROL_DIR / "attention_weights" / f"{run_id}.csv"
        atomic_csv_dump(pd.DataFrame(attn_rows), attn_path)

    elapsed = time.time() - t0
    run_grad_stats = summarize_preclip_grad_norms(all_grad_norms)
    final_row = hist_df.iloc[-1]
    rec = {
        "protocol_hash": RANDOM_INIT_CONTROL_PROTOCOL_HASH,
        "base_protocol_hash": PROTOCOL_HASH,
        "ft_protocol_hash_reference": FT_PROTOCOL_HASH,
        "family": "RANDOM_INIT",
        "topic": topic,
        "arch": arch,
        "learning_rate": RANDOM_INIT_LEARNING_RATE,
        "weight_decay": RANDOM_INIT_WEIGHT_DECAY,
        "master_seed": int(master_seed),
        "derived_seeds": seeds,
        "encoder_init_seed": int(enc_seed),
        "same_encoder_seed_across_architectures_within_topic": True,
        "run_id": run_id,
        "best_epoch": int(best_epoch),
        "epochs_ran": int(len(history)),
        "fixed_epoch_budget": int(RANDOM_INIT_MAX_EPOCHS),
        "early_stopping_used": False,
        "best_val_weighted_macro_f1": best_metrics["weighted_macro_f1"],
        "best_val_weighted_accuracy": best_metrics["weighted_accuracy"],
        "best_val_weighted_cross_entropy": best_metrics["weighted_cross_entropy"],
        "best_val_raw_macro_f1": best_metrics["raw_macro_f1"],
        "final_epoch_val_weighted_macro_f1": float(final_row.val_weighted_macro_f1),
        "final_epoch_val_weighted_accuracy": float(final_row.val_weighted_accuracy),
        "final_epoch_val_weighted_cross_entropy": float(final_row.val_weighted_cross_entropy),
        "final_epoch_val_raw_macro_f1": float(final_row.val_raw_macro_f1),
        "runtime_seconds": float(elapsed),
        "history_path": str(hist_path.relative_to(PROJECT_ROOT)),
        "checkpoint_path": str(checkpoint_path.relative_to(PROJECT_ROOT)),
        "validation_predictions_path": str(pred_path.relative_to(PROJECT_ROOT)),
        "attention_weights_path": str(attn_path.relative_to(PROJECT_ROOT)) if attn_path else None,
        "steps_per_epoch": int(steps_per_epoch),
        "max_scheduler_steps": int(total_steps),
        "warmup_steps": int(warmup_steps),
        "warmup_ratio": float(RANDOM_INIT_WARMUP_RATIO),
        "skipped_amp_optimizer_steps": int(skipped_steps),
        "mean_train_phase_seconds": float(hist_df.train_phase_seconds.mean()),
        "mean_train_eval_seconds": float(hist_df.train_eval_seconds.mean()),
        "mean_val_eval_seconds": float(hist_df.val_eval_seconds.mean()),
        "mean_epoch_seconds": float(hist_df.epoch_seconds.mean()),
        "max_epoch_seconds": float(hist_df.epoch_seconds.max()),
        **run_grad_stats,
        "gradient_clipping_max_norm": MAX_GRAD_NORM,
        "ft512_physical_batch_size": FT512_PHYSICAL_BATCH_SIZE if arch == "512" else None,
        "gradient_checkpointing": bool(_use_gc),
        "hier_train_method": HIER_TRAIN_METHOD if arch != "512" else None,
        "hier_chunk_microbatch_size": HIER_CHUNK_MICROBATCH_SIZE if arch != "512" else None,
        "runtime_status_path": str(runtime_status_path.relative_to(PROJECT_ROOT)),
        "test_evaluated": False,
    }
    atomic_json_dump(rec, record_path)
    atomic_json_dump({
        "family": "RANDOM_INIT",
        "topic": topic,
        "arch": arch,
        "protocol_hash": RANDOM_INIT_CONTROL_PROTOCOL_HASH,
        "base_protocol_hash": PROTOCOL_HASH,
        "ft_protocol_hash_reference": FT_PROTOCOL_HASH,
        "single_predefined_hyperparameter_setting": True,
        "master_seed": int(master_seed),
        "encoder_init_seed": int(enc_seed),
        "checkpoint_path": str(checkpoint_path.relative_to(PROJECT_ROOT)),
        "run_record_path": str(record_path.relative_to(PROJECT_ROOT)),
        "best_epoch": int(best_epoch),
    }, selected_dir / "metadata.json")

    _write_runtime_status(
        "complete", RANDOM_INIT_MAX_EPOCHS,
        runtime_seconds=float(elapsed),
        best_epoch=int(best_epoch),
        best_val_weighted_macro_f1=float(best_metrics["weighted_macro_f1"]),
    )
    if VERBOSE_EPOCHS:
        print(f"{run_id} FINALIZE done", flush=True)

    del model, optimizer, scheduler, scaler, best_state
    cuda_cleanup()
    return rec

print("Random-init control protocol hash:", RANDOM_INIT_CONTROL_PROTOCOL_HASH)
print("Random-init fixed setting:", {
    "lr": RANDOM_INIT_LEARNING_RATE,
    "wd": RANDOM_INIT_WEIGHT_DECAY,
    "epochs": RANDOM_INIT_MAX_EPOCHS,
    "warmup_ratio": RANDOM_INIT_WARMUP_RATIO,
    "master_seed": RANDOM_INIT_MASTER_SEED,
    "expected_runs": RANDOM_INIT_EXPECTED_RUNS,
})

Random-init control protocol hash: 9efe780c0f50e591a04e6c9e4212fd5e3b8f1cc52a456fd9d1def93127f966d1
Random-init fixed setting: {'lr': 0.0001, 'wd': 0.01, 'epochs': 100, 'warmup_ratio': 0.1, 'master_seed': 20260904, 'expected_runs': 6}


In [16]:
# 16) Run/reuse the six random-initialized controls and save their validation summary
random_init_records = []
for topic in TOPICS:
    for arch in ARCHITECTURES:
        print("RANDOM_INIT", topic, arch)
        random_init_records.append(train_random_init_run(topic, arch, RANDOM_INIT_MASTER_SEED))

_random_init_candidate_df = pd.DataFrame(random_init_records)
_random_init_all_path = RANDOM_INIT_CONTROL_DIR / "all_results.csv"

# On resume, keep the already-completed aggregate table instead of reserializing
# JSON-loaded records with a different dictionary/column order.
if _random_init_all_path.exists():
    _existing_random_init_df = pd.read_csv(_random_init_all_path)
    if (
        len(_existing_random_init_df) == RANDOM_INIT_EXPECTED_RUNS
        and set(_existing_random_init_df["run_id"]) == set(_random_init_candidate_df["run_id"])
    ):
        random_init_df = _existing_random_init_df
    else:
        random_init_df = _random_init_candidate_df
        atomic_csv_dump(random_init_df, _random_init_all_path)
else:
    random_init_df = _random_init_candidate_df
    atomic_csv_dump(random_init_df, _random_init_all_path)

assert len(random_init_df) == RANDOM_INIT_EXPECTED_RUNS == 6
assert (random_init_df.epochs_ran == RANDOM_INIT_MAX_EPOCHS).all()
assert (~random_init_df.early_stopping_used).all()
assert not random_init_df.test_evaluated.any()

# Verify that the three architectures within each topic used the same dedicated random encoder seed.
assert (random_init_df.groupby("topic").encoder_init_seed.nunique() == 1).all()
random_init_summary_cols = [
    "topic", "arch", "family", "learning_rate", "weight_decay", "master_seed", "encoder_init_seed",
    "best_epoch", "best_val_weighted_macro_f1", "best_val_weighted_accuracy",
    "best_val_weighted_cross_entropy", "best_val_raw_macro_f1",
    "final_epoch_val_weighted_macro_f1", "final_epoch_val_weighted_accuracy",
    "final_epoch_val_weighted_cross_entropy", "final_epoch_val_raw_macro_f1",
    "runtime_seconds", "skipped_amp_optimizer_steps", "grad_clip_fraction",
    "preclip_grad_norm_mean", "preclip_grad_norm_max", "checkpoint_path",
]
random_init_summary_df = random_init_df[random_init_summary_cols].copy()
atomic_csv_dump(random_init_summary_df, OUTPUT_DIR / "RANDOM_INIT_CONTROL_VALIDATION_SUMMARY.csv")
display(random_init_summary_df.sort_values(["topic", "arch"]))
print("Random-initialized RoBERTa control complete; test remains sealed.")

RANDOM_INIT topicA 512
RANDOM_INIT topicA MEAN
RANDOM_INIT topicA ATTN
RANDOM_INIT topicB 512
RANDOM_INIT topicB MEAN
RANDOM_INIT topicB ATTN


,topic,arch,family,learning_rate,weight_decay,master_seed,encoder_init_seed,best_epoch,best_val_weighted_macro_f1,best_val_weighted_accuracy,...,final_epoch_val_weighted_macro_f1,final_epoch_val_weighted_accuracy,final_epoch_val_weighted_cross_entropy,final_epoch_val_raw_macro_f1,runtime_seconds,skipped_amp_optimizer_steps,grad_clip_fraction,preclip_grad_norm_mean,preclip_grad_norm_max,checkpoint_path
0,topicA,512,RANDOM_INIT,0.0001,0.01,20260904,2882082750,2,0.362716,0.461111,...,0.166667,0.333333,1.098624,0.193211,2188.047716,2,1.00000,6.270770,318.570129,modeling_results/roberta_transformer_final/sel...
2,topicA,ATTN,RANDOM_INIT,0.0001,0.01,20260904,2882082750,7,0.167794,0.333333,...,0.166667,0.333333,1.098638,0.193211,6388.486066,0,1.00000,5.974759,37.103493,modeling_results/roberta_transformer_final/sel...
1,topicA,MEAN,RANDOM_INIT,0.0001,0.01,20260904,2882082750,63,0.738778,0.736201,...,0.720643,0.720609,2.413717,0.732599,6412.252349,2,0.41907,3.767995,150.763885,modeling_results/roberta_transformer_final/sel...
3,topicB,512,RANDOM_INIT,0.0001,0.01,20260904,1014125518,40,0.521822,0.621622,...,0.517120,0.615616,0.648828,0.570299,1535.479773,0,1.00000,5.260482,54.236610,modeling_results/roberta_transformer_final/sel...
5,topicB,ATTN,RANDOM_INIT,0.0001,0.01,20260904,1014125518,11,0.542363,0.648649,...,0.166667,0.333333,1.098614,0.089517,4137.996824,0,1.00000,6.335661,60.173813,modeling_results/roberta_transformer_final/sel...
4,topicB,MEAN,RANDOM_INIT,0.0001,0.01,20260904,1014125518,11,0.542363,0.648649,...,0.166667,0.333333,1.098613,0.089517,4182.231019,0,1.00000,6.327048,103.937080,modeling_results/roberta_transformer_final/sel...


Random-initialized RoBERTa control complete; test remains sealed.


## Final summaries

The final cells save compact validation summaries and plots for the selected FE, FT and random-initialized models.

In [17]:
# 17) Final selected-model/control summaries and plots

def fe_selected_seed_runs(topic: str, arch: str) -> pd.DataFrame:
    cfg=FE_SELECTED[topic][arch]["config_id"]
    return fe2_df[(fe2_df.topic==topic)&(fe2_df.arch==arch)&(fe2_df.config_id==cfg)].copy()


def ft_selected_seed_runs(topic: str, arch: str) -> pd.DataFrame:
    meta=json_load(OUTPUT_DIR/"selected_models"/topic/f"FT-{arch}"/"metadata.json")
    cfg=meta["selected_config"]["config_id"]
    df=pd.DataFrame([json_load(p) for p in (FT_STAGE2_DIR/"run_records").glob("*.json")])
    return df[(df.topic==topic)&(df.arch==arch)&(df.config_id==cfg)].copy()


selected_summary_rows=[]
for topic in TOPICS:
    for family in ["FE","FT"]:
        for arch in ARCHITECTURES:
            runs=fe_selected_seed_runs(topic,arch) if family=="FE" else ft_selected_seed_runs(topic,arch)
            selected_summary_rows.append({
                "topic":topic,"family":family,"arch":arch,
                "n_confirmation_seeds":int(len(runs)),
                "learning_rate":float(runs.learning_rate.iloc[0]),"weight_decay":float(runs.weight_decay.iloc[0]),
                "mean_best_val_weighted_macro_f1":float(runs.best_val_weighted_macro_f1.mean()),
                "std_best_val_weighted_macro_f1":float(runs.best_val_weighted_macro_f1.std(ddof=1)),
                "mean_best_val_weighted_accuracy":float(runs.best_val_weighted_accuracy.mean()),
                "mean_best_val_weighted_cross_entropy":float(runs.best_val_weighted_cross_entropy.mean()),
                "mean_best_epoch":float(runs.best_epoch.mean()),
                "mean_runtime_seconds":float(runs.runtime_seconds.mean()),
                "mean_grad_clip_fraction":float(runs.grad_clip_fraction.mean()),
                "mean_preclip_grad_norm":float(runs.preclip_grad_norm_mean.mean()),
                "max_preclip_grad_norm_across_runs":float(runs.preclip_grad_norm_max.max()),
                "mean_skipped_amp_optimizer_steps":(
                    float(runs.skipped_amp_optimizer_steps.mean())
                    if family == "FT" and "skipped_amp_optimizer_steps" in runs.columns else 0.0
                ),
            })
selected_summary_df=pd.DataFrame(selected_summary_rows)
atomic_csv_dump(selected_summary_df,OUTPUT_DIR/"SELECTED_MODEL_VALIDATION_SUMMARY.csv")
display(selected_summary_df)

# Existing FE/FT selected-model comparison remains separate because RANDOM_INIT is a single-seed control,
# not a confirmation-selected family.
for topic in TOPICS:
    sub=selected_summary_df[selected_summary_df.topic==topic].copy()
    labels=[f"{r.family}-{r.arch}" for r in sub.itertuples()]
    x=np.arange(len(sub))
    fig,ax=plt.subplots(figsize=(9,4.8))
    ax.bar(x,sub.mean_best_val_weighted_macro_f1,yerr=sub.std_best_val_weighted_macro_f1,capsize=4)
    ax.set_xticks(x,labels,rotation=30,ha="right")
    ax.set_ylabel("Validation weighted macro-F1")
    ax.set_title(f"Selected transformer models — {topic} (confirmation mean ± SD; FE n=5, FT n=3)")
    save_figure(fig,OUTPUT_DIR/"plots"/f"selected_models_{topic}_validation_f1.png")

# Direct architecture-matched FT-vs-random-initialization validation comparison.
for topic in TOPICS:
    ft_rows = selected_summary_df[(selected_summary_df.topic==topic)&(selected_summary_df.family=="FT")].set_index("arch")
    ri_rows = random_init_summary_df[random_init_summary_df.topic==topic].set_index("arch")
    labels=[]; vals=[]; errs=[]
    for arch in ARCHITECTURES:
        labels.extend([f"FT-{arch}\nmean n=3", f"Random-{arch}\nn=1"])
        vals.extend([
            float(ft_rows.loc[arch,"mean_best_val_weighted_macro_f1"]),
            float(ri_rows.loc[arch,"best_val_weighted_macro_f1"]),
        ])
        errs.extend([
            float(ft_rows.loc[arch,"std_best_val_weighted_macro_f1"]),
            0.0,
        ])
    x=np.arange(len(labels))
    fig,ax=plt.subplots(figsize=(10,4.8))
    ax.bar(x,vals,yerr=errs,capsize=4)
    ax.set_xticks(x,labels,rotation=20,ha="right")
    ax.set_ylabel("Best validation weighted macro-F1")
    ax.set_title(f"Pretrained FT vs random-initialized RoBERTa control — {topic}")
    save_figure(fig,OUTPUT_DIR/"plots"/f"ft_vs_random_init_{topic}_validation_f1.png")

# Mean selected validation confusion matrices across confirmation seeds for FE/FT.
for topic in TOPICS:
    for family in ["FE","FT"]:
        for arch in ARCHITECTURES:
            runs=fe_selected_seed_runs(topic,arch) if family=="FE" else ft_selected_seed_runs(topic,arch)
            matrices=[]
            for rr in runs.itertuples(index=False):
                if family=="FE":
                    pred_path=FE_STAGE2_DIR/"validation_predictions"/f"{rr.run_id}.csv"
                else:
                    pred_path=PROJECT_ROOT/rr.validation_predictions_path
                pdf=pd.read_csv(pred_path)
                matrices.append(confusion_matrix(pdf.y_true,pdf.y_pred,labels=np.arange(3),sample_weight=pdf.balance_weight))
            cm=np.mean(matrices,axis=0)
            fig,ax=plt.subplots(figsize=(5.2,4.5))
            im=ax.imshow(cm)
            ax.set_xticks(range(3),PORTALS,rotation=30,ha="right"); ax.set_yticks(range(3),PORTALS)
            ax.set_xlabel("Predicted"); ax.set_ylabel("True")
            ax.set_title(f"{topic} {family}-{arch}: mean weighted val confusion")
            for i in range(3):
                for j in range(3): ax.text(j,i,f"{cm[i,j]:.1f}",ha="center",va="center")
            fig.colorbar(im,ax=ax,fraction=0.046,pad=0.04)
            save_figure(fig,OUTPUT_DIR/"plots"/f"{topic}_{family}_{arch}_mean_weighted_val_confusion.png")

# Single-run weighted validation confusion matrices for RANDOM_INIT controls.
for rr in random_init_df.itertuples(index=False):
    pdf=pd.read_csv(PROJECT_ROOT/rr.validation_predictions_path)
    cm=confusion_matrix(pdf.y_true,pdf.y_pred,labels=np.arange(3),sample_weight=pdf.balance_weight)
    fig,ax=plt.subplots(figsize=(5.2,4.5))
    im=ax.imshow(cm)
    ax.set_xticks(range(3),PORTALS,rotation=30,ha="right"); ax.set_yticks(range(3),PORTALS)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{rr.topic} RANDOM-{rr.arch}: weighted val confusion (n=1)")
    for i in range(3):
        for j in range(3): ax.text(j,i,f"{cm[i,j]:.1f}",ha="center",va="center")
    fig.colorbar(im,ax=ax,fraction=0.046,pad=0.04)
    save_figure(fig,OUTPUT_DIR/"plots"/f"{rr.topic}_RANDOM_{rr.arch}_weighted_val_confusion.png")

print("Transformer model-selection summaries saved to", OUTPUT_DIR)


,topic,family,arch,n_confirmation_seeds,learning_rate,weight_decay,mean_best_val_weighted_macro_f1,std_best_val_weighted_macro_f1,mean_best_val_weighted_accuracy,mean_best_val_weighted_cross_entropy,mean_best_epoch,mean_runtime_seconds,mean_grad_clip_fraction,mean_preclip_grad_norm,max_preclip_grad_norm_across_runs,mean_skipped_amp_optimizer_steps
0,topicA,FE,512,5,0.00300,0.00,0.696092,0.031050,0.696344,0.823480,28.600000,3.999031,0.811672,1.976902,7.042555,0.000000
1,topicA,FE,MEAN,5,0.01000,0.00,0.778657,0.027964,0.777384,0.641188,38.400000,10.981763,0.810713,1.968777,6.870336,0.000000
2,topicA,FE,ATTN,5,0.01000,0.00,0.708320,0.010358,0.707061,0.760140,21.800000,9.736124,0.798423,1.926131,6.864344,0.000000
3,topicA,FT,512,3,0.00001,0.01,0.939886,0.006361,0.940024,0.236379,15.333333,415.749544,0.755867,20.341374,230.488327,5.000000
4,topicA,FT,MEAN,3,0.00001,0.00,0.947256,0.019634,0.947342,0.170520,12.000000,1102.799077,0.853750,17.923734,192.037308,3.666667
5,topicA,FT,ATTN,3,0.00001,0.01,0.953912,0.004380,0.953943,0.196040,17.000000,1333.699931,0.741998,14.298694,162.987473,3.666667
6,topicB,FE,512,5,0.01000,0.01,0.741786,0.012181,0.745314,0.633456,39.800000,3.534970,0.780430,1.879787,8.941511,0.000000
7,topicB,FE,MEAN,5,0.01000,0.00,0.768783,0.013855,0.771838,0.596103,38.400000,7.724817,0.774508,1.864432,9.063279,0.000000
8,topicB,FE,ATTN,5,0.01000,0.01,0.707366,0.019773,0.712583,0.697491,24.200000,7.461946,0.782301,1.871713,9.051895,0.000000
9,topicB,FT,512,3,0.00002,0.01,0.941312,0.015696,0.940994,0.360138,17.000000,336.543063,0.641498,12.646883,182.388687,4.333333


Transformer model-selection summaries saved to /home/dorjan/NLP/nlp_portal_project/modeling_results/roberta_transformer_final


## Saved training histories

Completed run records and checkpoints are reused automatically when the notebook is run again.

In [18]:
# Optional helper for saved training histories

def show_history(run_id: str):
    matches=list(OUTPUT_DIR.rglob(f"{run_id}.csv"))
    matches=[p for p in matches if p.parent.name=="histories"]
    if not matches:
        raise FileNotFoundError(run_id)
    df=pd.read_csv(matches[0])
    display(df)
    fig,ax=plt.subplots(figsize=(8,4.5))
    ax.plot(df.epoch,df.full_train_weighted_macro_f1,label="full train weighted F1")
    ax.plot(df.epoch,df.val_weighted_macro_f1,label="validation weighted F1")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Macro-F1"); ax.legend(); ax.set_title(run_id)
    plt.show()
    return df

print("Training-history helper ready.")


Training-history helper ready.
